# D45 Dataset Automation Code

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 파인튜닝 데이터셋 생성 자동화 실습

## 실행 환경

**실습 목표**
- 난이도와 프롬프트 변주를 적용해 공개 가중치 모델로 회의록 정리 합성 데이터를 생성하고 1차 규칙 필터(형식·원문 대조·근사 중복)와 검색 왕복 검사를 구현할 수 있다.
- 이종 Judge 모델(`Llama-3.2-3B-Instruct`) 기반 2차 판정, 난이도 분포 재조정, 3차 층화 표본 추출 및 인적 검수 결과의 1차 필터 규칙 환류를 수행할 수 있다.
- 회의록 출처 단위로 훈련·테스트 세트를 분할해 데이터셋 카드를 작성하고 동일 건수(50건) SFT 학습 A/B 비교로 큐레이션 효과를 정량 증명할 수 있다.

<div style="font-size: 18px; line-height: 1.6;">

| 항목 | 값 |
|---|---|
| 런타임 | 코랩 Pro+ / GPU. T4(16GB)에서 끝까지 실행됩니다 |
| 필요한 키 | 없습니다. 게이트가 걸리지 않은 공개 모델만 씁니다 |
| 로드하는 모델 | 3개. 생성 3B(4bit), 판정 3B(4bit), 학습 베이스 1.5B(4bit) (총 VRAM 약 6GB) |
| 디스크 | 받아 두는 가중치 합계 약 12GB |

</div>

<div style="font-size: 18px; line-height: 1.6;">

| 오래 걸리는 셀 | T4 기준 대략 |
|---|---|
| Step 3 생성 240건 | 25\~40분 |
| Step 4.4 judge 모델 로드와 판정 | 10\~15분 |
| Step 5 분포 재조정 (추가 생성) | 5\~15분 |
| Step 8 A/B 학습 두 번 | 각 15\~25분 |
| 선택 트랙 1 (학습 여섯 번) | 60\~90분 |
| 선택 트랙 6 (생성 약 50회, 어림값) | 5\~10분 |

</div>

**[현업 환경 적용]** 표시가 붙은 곳은 **코랩에서는 흉내만 내고 사내에서는 달라지는 부분**입니다. 시드를 사내 회의록으로 바꾸는 곳(Step 2), judge 모델 가중치를 승인받아 사내 저장소에서 읽는 곳과 gold 대신 사람이 판정한 표본을 쓰는 곳(Step 4.4), 조건 확인 기록과 데이터 시점 기록을 사내 기록으로 바꾸는 곳(Step 7)입니다.

오늘은 생성과 판정과 학습이 전부 모델 위에서 실행되므로 GPU가 없으면 Step 3부터 막힙니다. Step 0을 실행하기 전에 런타임 유형을 GPU로 바꿉니다.

## Step 0. 환경 준비

메이저 버전이 올라가면 인자 이름이 바뀌어 예제가 통째로 실행되지 않는 분야라 메이저 버전에 상한을 두는 방식(`"이름>=x,<y"`)으로 버전을 고정해 설치합니다.

In [1]:
!pip install -q "transformers>=5.17,<6" "accelerate>=1.15,<2" "bitsandbytes>=0.50,<0.51"
!pip install -q "peft>=0.21,<0.22" "trl>=1.13,<2" "datasets>=5.0,<6"
!pip install -q "datasketch>=2,<3" "langchain-huggingface>=1.2,<2" "sentence-transformers>=6,<7"

# 설치 뒤 런타임을 한 번 재시작하세요.
# transformers 를 고정하면 함께 설치되는 huggingface-hub 버전이 바뀌어, 재시작 전에는 옛 버전이 메모리에 남습니다.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [2]:
# 실제로 어떤 버전이 설치되었는지 출력해 기록해 둡니다. 나중에 결과가 달라지면 여기부터 봅니다
import importlib.metadata as md

for name in ["torch", "transformers", "peft", "trl", "datasets",
             "accelerate", "bitsandbytes", "datasketch"]:
    try:
        print(f"{name:16} {md.version(name)}")
    except md.PackageNotFoundError:
        print(f"{name:16} (설치되지 않음)")

torch            2.11.0+cu130
transformers     5.20.2
peft             0.21.2
trl              1.16.0
datasets         5.1.0
accelerate       1.15.0
bitsandbytes     0.50.2
datasketch       2.0.0


`__version__` 속성 대신 `importlib.metadata.version()`을 썼습니다. **패키지 중에는 `__version__`을 아예 두지 않는 것이 있어서** 설치 정보에서 읽는 편이 어떤 패키지에나 통합니다.

이 출력은 데이터셋 카드에 그대로 옮겨 적습니다. 반년 뒤에 같은 코드가 다른 결과를 내면 가장 먼저 볼 기록입니다.

In [3]:
import os, re, json, time, random, hashlib, gc
from pathlib import Path
from collections import Counter
import numpy as np
import torch

random.seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU : {torch.cuda.get_device_name(0)}  VRAM {props.total_memory/1e9:.1f} GB")
else:
    print("GPU 없음. Step 3 부터 모델을 올리므로 런타임 유형을 GPU로 바꾸세요.")

GPU : Tesla T4  VRAM 15.6 GB


`torch.cuda.get_device_name(0)`을 조건 없이 호출하면 **GPU가 없는 런타임에서 첫 셀부터 예외가 발생하므로** `torch.cuda.is_available()`로 먼저 확인한 뒤 안내 메시지를 출력합니다. 이론 노트 전반에서 강조한 **'실패를 미처리 예외가 아닌 정상 제어 흐름으로 다룬다'** 는 원칙을 코드로 구현한 것이며 Step 4.4에서 Judge 모델을 로드할 때도 동일하게 적용합니다.

## Step 1. 회의록 표준 양식 변환 태스크 및 프롬프트 정의

교재 : 4.5.3 파인 튜닝 데이터셋 생성

이론 1.6절의 네 항목을 명시적으로 정의하고 특히 **성공 기준을 기계 검사 가능한 형태로** 씁니다.

In [4]:
TASK = {
    "name": "회의록 표준 양식 정리",
    "input_form": "자유 형식 회의록 원문 (2~6개 안건, 참석자 3~6명)",
    "output_form": "4개 섹션 마크다운 (회의 개요 / 결정 사항 / 논의 사항 / 후속 조치)",
    "success_criteria": [
        "네 개 섹션 헤더가 모두 존재한다",
        "결정 사항의 각 항목에 담당과 기한이 표기된다",
        "일시와 참석자가 원문과 일치한다",
        "결정된 것과 논의만 된 것이 올바르게 분리된다",
    ],
    "failure_modes": [
        "원문에 없는 내용을 추가한다",
        "결정과 논의를 구분하지 못한다",
        "담당자나 기한을 임의로 만들어낸다",
        "양식은 지켰으나 내용이 비어 있다",
    ],
}
print(json.dumps(TASK, ensure_ascii=False, indent=2))

{
  "name": "회의록 표준 양식 정리",
  "input_form": "자유 형식 회의록 원문 (2~6개 안건, 참석자 3~6명)",
  "output_form": "4개 섹션 마크다운 (회의 개요 / 결정 사항 / 논의 사항 / 후속 조치)",
  "success_criteria": [
    "네 개 섹션 헤더가 모두 존재한다",
    "결정 사항의 각 항목에 담당과 기한이 표기된다",
    "일시와 참석자가 원문과 일치한다",
    "결정된 것과 논의만 된 것이 올바르게 분리된다"
  ],
  "failure_modes": [
    "원문에 없는 내용을 추가한다",
    "결정과 논의를 구분하지 못한다",
    "담당자나 기한을 임의로 만들어낸다",
    "양식은 지켰으나 내용이 비어 있다"
  ]
}


## Step 2. 난이도별 회의록 원문 시드 데이터 생성

교재 : 4.3.1 합성 데이터

실무에서는 사내 회의록을 쓰지만 실습에서는 규칙으로 원문을 만들고 **난이도를 통제할 수 있게** 설계합니다. 규칙으로 만든 원문도 합성 데이터입니다(데이터 엔지니어링 노트 6.4절).

이론 4.5절의 첫 번째 방법인 **입력의 특징으로 난이도를 나누기**를 안건 수, 미지정 항목, 결정과 논의의 혼재, 조건부 기한으로 구현합니다.

같은 코드가 이론 2.6절의 두 번째 설정인 **입력 흔들기**이기도 합니다. 길이와 항목 수와 빠진 정보의 개수가 매번 달라져 출력이 서로 덜 닮습니다.

**[현업 환경 적용]** : 사내에서는 여기에 실제 회의록을 넣습니다. 그때 난이도는 `difficulty` 인자가 아니라 **원문에서 세어 붙입니다.** 안건 수와 "미정"의 개수를 세는 함수 하나면 됩니다.

In [5]:
TOPICS = ["신규 기능 출시", "인프라 비용 절감", "채용 계획", "보안 감사 대응",
          "고객 이슈 처리", "분기 실적 검토", "협력사 계약", "교육 개편",
          "데이터 이관", "품질 지표 개선"]
PEOPLE = ["김서준", "이지우", "박민재", "최유나", "정하윤", "강도현", "윤서아",
          "임태호", "한소율", "오재현"]
TEAMS = ["개발1팀", "개발2팀", "인프라팀", "기획팀", "영업팀", "보안팀", "인사팀"]


def make_minutes(seed, difficulty="normal"):
    """난이도를 통제해 회의록 원문과 정답 구조를 함께 생성합니다"""
    rnd = random.Random(seed)
    cfg = {
        "easy": {"items": (2, 2), "missing_owner": 0.0, "conditional": 0.0, "mixed": False},
        "normal": {"items": (3, 4), "missing_owner": 0.25, "conditional": 0.15, "mixed": False},
        "hard": {"items": (4, 5), "missing_owner": 0.5, "conditional": 0.4, "mixed": True},
    }[difficulty]

    topic = rnd.choice(TOPICS)
    attendees = rnd.sample(PEOPLE, rnd.randint(3, 5))
    month, day = rnd.randint(1, 12), rnd.randint(1, 28)
    n_items = rnd.randint(*cfg["items"])

    lines = [f"{month}월 {day}일 {topic} 회의. 참석 {', '.join(attendees)}."]
    decided, discussed = [], []

    for i in range(1, n_items + 1):
        # 최소 1개 안건은 결정 사항이 되도록 보장합니다 (결정 사항이 0개면 양식 필터에서 탈락하므로)
        if i == n_items and not decided:
            is_decision = True
        else:
            is_decision = rnd.random() > (0.5 if cfg["mixed"] else 0.35)

        if is_decision:
            owner = "미정" if rnd.random() < cfg["missing_owner"] else rnd.choice(attendees)
            if rnd.random() < cfg["conditional"]:
                deadline = f"{month+1 if month < 12 else 1}월 중 (예산 확정 후)"
            else:
                deadline = "미정" if owner == "미정" else \
                    f"{month+1 if month < 12 else 1}월 {rnd.randint(1,28)}일"
            what = f"{topic} {i}번 안건 추진"
            owner_txt = "담당자 미정" if owner == "미정" else f"{owner} 담당"
            lines.append(f"- {i}. {what} 확정. {owner_txt}, 기한 {deadline}.")
            decided.append({"what": what, "owner": owner, "deadline": deadline})
        else:
            what = f"{topic} {i}번 안건"
            reason = rnd.choice(["추가 검토 필요", "관련 부서 의견 수렴 후 재논의",
                                 "예산 확인 후 결정", "다음 회의로 이월"])
            lines.append(f"- {i}. {what}은 {reason}. {rnd.choice(TEAMS)} 협의 예정.")
            discussed.append({"what": what, "reason": reason})

    return {
        "raw": "\n".join(lines),
        "gold": {"date": f"{month}월 {day}일", "attendees": attendees, "topic": topic,
                 "decided": decided, "discussed": discussed},
        "difficulty": difficulty,
    }


for d in ["easy", "hard"]:
    m = make_minutes(1, d)
    print(f"=== {d} ===")
    print(m["raw"])
    print()

=== easy ===
8월 21일 채용 계획 회의. 참석 이지우, 정하윤, 오재현, 최유나, 윤서아.
- 1. 채용 계획 1번 안건 추진 확정. 이지우 담당, 기한 9월 13일.
- 2. 채용 계획 2번 안건 추진 확정. 이지우 담당, 기한 9월 9일.

=== hard ===
8월 21일 채용 계획 회의. 참석 이지우, 정하윤, 오재현, 최유나, 윤서아.
- 1. 채용 계획 1번 안건 추진 확정. 담당자 미정, 기한 9월 중 (예산 확정 후).
- 2. 채용 계획 2번 안건 추진 확정. 담당자 미정, 기한 미정.
- 3. 채용 계획 3번 안건은 다음 회의로 이월. 인프라팀 협의 예정.
- 4. 채용 계획 4번 안건 추진 확정. 담당자 미정, 기한 미정.
- 5. 채용 계획 5번 안건 추진 확정. 담당자 미정, 기한 9월 중 (예산 확정 후).



## Step 3. 공개 가중치 모델 기반 합성 데이터 생성 파이프라인

교재 : 4.5.3 파인 튜닝 데이터셋 생성, 4.5.4 지시 파인 튜닝 데이터셋 자동 생성

사내 GPU에 올린 공개 가중치 모델로 정리본을 생성합니다. 이론 5.2절의 첫 번째 질문에서 **"우리 GPU에 올린 공개 가중치"** 쪽을 고른 것이라 볼 조건 문서가 하나로 줄어듭니다. 항목마다 `instruction`(지시 프롬프트), `input`(원문), `output`(정리본)을 담습니다. 필드 이름이 교재 4.5.4가 예로 든 JSON 형식과 같습니다. Step 7에서 이 세 필드만 따로 내보냅니다.

모델 선택 노트에서 정한 대로 코랩 한 장에 세 개를 올릴 수 있고 게이트와 미러 상황이 확인된 **한두 세대 앞의 버전**을 씁니다. 사내에서는 같은 코드에 현행 버전의 저장소 이름만 넣으면 됩니다.

In [6]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

transformers.logging.set_verbosity_error()  # 반복적인 generation_config / tokenizer 경고 숨김

GEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
gen_tok = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    quantization_config=bnb,
    dtype=torch.float16,          # torch_dtype= 는 폐기되었습니다
    device_map="auto",
)
gen_model.eval()
if getattr(gen_model, "generation_config", None) is not None:
    gen_model.generation_config.max_length = None

# 어떤 버전을 받았는지 기록해 둡니다. 데이터셋 카드의 "조건 확인"에 그대로 들어갑니다
GEN_REVISION = getattr(gen_model.config, "_commit_hash", None) or "(확인 못 함)"
print(f"생성 모델 로드 {GEN_MODEL} / 버전 {GEN_REVISION}")
print(f"VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

생성 모델 로드 Qwen/Qwen2.5-3B-Instruct / 버전 (확인 못 함)
VRAM 2.06 GB


`dtype=`을 썼습니다. 예전의 `torch_dtype=`은 폐기되어 경고가 나고 다음 메이저 버전에서 없어집니다.

`_commit_hash`도 읽어 두었습니다. 모델 이름만 적으면 **같은 이름으로 가중치가 바뀐 경우**를 구분할 수 없습니다. 그래서 이론 5.3절에서 이름과 버전 번호를 함께 적으라고 했습니다.

In [7]:
import transformers
transformers.logging.set_verbosity_error()

SYSTEM_PROMPT = (
    "당신은 사내 회의록을 표준 양식으로 정리하는 담당자입니다. "
    "요청한 양식만 출력하고, 답변은 반드시 한국어로 작성합니다."
)


def chat(model, tok, prompt, system=SYSTEM_PROMPT, max_new_tokens=320, temperature=0.7):
    """모델 하나에 한 번 묻습니다. 뒤에서 judge 모델에도 같은 함수를 씁니다"""
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    ids = tok(text, return_tensors="pt").to(model.device)

    kw = ({"do_sample": True, "temperature": temperature, "top_p": 0.9}
          if temperature > 0 else {"do_sample": False})

    if getattr(model, "generation_config", None) is not None:
        model.generation_config.max_length = None

    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens,
                             pad_token_id=tok.eos_token_id, **kw)
    return tok.decode(out[0][ids["input_ids"].shape[1]:],
                      skip_special_tokens=True,
                      clean_up_tokenization_spaces=False)


def generate(prompt, **kw):
    """생성 모델로 정리본을 만듭니다"""
    return chat(gen_model, gen_tok, prompt, **kw)

시스템 프롬프트에 **"답변은 반드시 한국어로 작성하세요"**를 명시했습니다. 사전학습 코퍼스의 상당 부분이 영어와 중국어로 구성된 다국어 모델은 한국어 원문을 입력받아도 프롬프트가 길어지면 영어나 중국어가 섞여 나오기 쉬우므로 출력 언어를 명시적으로 고정해야 합니다.

In [8]:
FORMAT_SPEC = """## 회의 개요
- 일시 :
- 참석 :
- 주제 :

## 결정 사항
1. (내용) / 담당: (담당자 또는 미정) / 기한: (기한 또는 미정)

## 논의 사항
- (결론이 나지 않은 안건): (사유)

## 후속 조치
- (다음 회의까지 할 일)"""

GEN_PROMPT = """다음 회의록 원문을 아래 표준 양식으로 정리하세요.

[표준 양식]
{spec}

[규칙]
1. 확정된 안건만 결정 사항에 넣고, 검토나 재논의가 필요한 안건은 논의 사항에 넣으세요.
2. 결정 사항의 각 항목에 담당과 기한을 반드시 표기하세요. 원문에 없으면 '미정'으로 적으세요.
3. 원문에 없는 내용을 절대 추가하지 마세요. 인명, 날짜, 숫자를 만들어내지 마세요.
4. 일시와 참석자는 원문 그대로 옮기세요.
5. 정리본만 출력하고 설명을 붙이지 마세요.

[회의록 원문]
{raw}"""

# 이론 2.6절 세 설정 중 1번 : 같은 일을 시키는 말을 네 가지로 바꿔 씁니다
INSTRUCTION_VARIANTS = [
    "다음 회의록 원문을 사내 표준 양식으로 정리하세요.",
    "아래 회의 기록을 표준 양식에 맞춰 정리해주세요.",
    "회의록을 표준 서식으로 재작성하세요.",
    "다음 회의 내용을 정해진 양식으로 옮겨주세요.",
]


# 실무 서빙 엔진(vLLM / OpenAI 호환 API) 연동 시 JSON 문법 오류를 0%로 만드는 Guided Decoding 스키마 명세 (이론 6.2절)
GUIDED_JSON_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "meeting_minutes_pair",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "overview": {"type": "string"},
                "decisions": {"type": "array", "items": {"type": "string"}},
                "discussions": {"type": "array", "items": {"type": "string"}},
                "actions": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["overview", "decisions", "discussions", "actions"],
            "additionalProperties": False,
        },
    },
}
print(f"프롬프트 변주 {len(INSTRUCTION_VARIANTS)}종 및 Guided Decoding JSON 스키마 준비 완료")

프롬프트 변주 4종 및 Guided Decoding JSON 스키마 준비 완료


**이론 2.6절의 설정이 있는 곳**

1. **프롬프트 변주** : `INSTRUCTION_VARIANTS` 목록을 사용합니다. 만드는 데 5분이 걸렸고 효과는 즉시 납니다  
2. **입력** : Step 2의 `difficulty`입니다. 안건 수와 빠진 정보의 개수가 매번 달라집니다. 이쪽이 더 크게 작용합니다  
3. **관점** : 오늘은 **쓰지 않습니다.** 출력 양식이 하나로 고정되어 누가 묻든 결과가 같아야 하는 태스크입니다. 같은 문서로 질의응답(QA) 쌍을 만드는 태스크라면 이 설정이 가장 크게 작용합니다

관점을 안 쓴 것도 판단입니다. **설정이 셋이라고 셋을 다 쓸 필요는 없습니다.** 다만 안 쓴 이유를 적어 둬야 나중에 다양성이 부족할 때 후보로 떠오릅니다.

In [9]:
# 난이도 분포를 정해 원본을 만듭니다. 이론 4.5절의 출발점 분포 (쉬움 3, 보통 5, 어려움 2)
# 실습 시간 단축을 위해 생성 규모를 60건(쉬움 18, 보통 30, 어려움 12)으로 경량화했습니다
TARGET_DIST = {"easy": 0.3, "normal": 0.5, "hard": 0.2}
N_GEN = 45

plan = []
for diff, ratio in TARGET_DIST.items():
    plan += [diff] * int(N_GEN * ratio)
random.shuffle(plan)
print(f"생성 계획 (총 {len(plan)}건) : {dict(Counter(plan))}")

생성 계획 (총 44건) : {'easy': 13, 'normal': 22, 'hard': 9}


In [10]:
# 생성 실행 (60건 경량화 기준 약 6~10분 소요)
raw_dataset = []
t0 = time.time()

for i, diff in enumerate(plan):
    m = make_minutes(1000 + i, diff)
    out = generate(GEN_PROMPT.format(spec=FORMAT_SPEC, raw=m["raw"]))
    raw_dataset.append({
        "id": i,
        "instruction": INSTRUCTION_VARIANTS[i % len(INSTRUCTION_VARIANTS)],
        "input": m["raw"],
        "output": out.strip(),
        "gold": m["gold"],
        "difficulty": diff,
        "source_seed": 1000 + i,
    })
    if (i + 1) % 10 == 0 or (i + 1) == len(plan):
        print(f"  {i+1}/{len(plan)} ({time.time()-t0:.0f}초)")

print(f"\n생성 완료 {len(raw_dataset)}건, {(time.time()-t0)/60:.1f}분")
print(raw_dataset[0]["output"][:400])

  10/44 (133초)
  20/44 (267초)
  30/44 (431초)
  40/44 (563초)
  44/44 (619초)

생성 완료 44건, 10.3분
## 회의 개요
- 일시 : 3월 18일
- 참석 : 이지우, 윤서아, 강도현, 김서준, 최유나
- 주제 : 협력사 계약 회의

## 결정 사항
1. 협력사 계약 2번 안건 추진 / 담당: 윤서아 / 기한: 4월 12일

## 논의 사항
- 협력사 계약 1번 안건: 관련 부서 의견 수렴 후 재논의 / 사유: 인프라팀 협의 예정

## 후속 조치
- 협력사 계약 1번 안건: 관련 부서 의견 수렴 후 재논의 / 사유: 인프라팀 혐의 예정
- 협력사 계약 2번 안건: 윤서아 담당, 기한: 4월 12일


## Step 4. 다단계 품질 필터링 구현

교재 : 4.5.4 지시 파인 튜닝 데이터셋 자동 생성, 4.4.4 데이터 중복 제거하기, 4.3.7 데이터 품질

이론 3.2절의 **1차 규칙 필터, 2차 LLM 판정, 3차 사람 검수**를 그대로 만듭니다.

<div style="font-size: 18px; line-height: 1.6;">

| 단계 | 어디에 있는가 | 무엇을 보는가 |
|---|---|---|
| 1차 규칙 필터 | 4.1 \~ 4.3 | 형식, 원문과의 대조, 출력끼리의 중복 |
| 2차 LLM 판정 | 4.4 | 형식은 맞는데 내용이 틀린 것 |
| 3차 사람 검수 | Step 6 | 1차와 2차가 놓친 것 |

</div>

비용 때문에 **싼 것부터** 합니다. 형식 검사는 문자열 몇 개를 찾으면 끝나지만 2차는 항목마다 모델을 한 번씩 호출합니다. 1차에서 탈락할 것을 모델 판정까지 보내면 그만큼 시간을 버립니다.

단계마다 **조치도 다릅니다.** 형식이 틀린 것은 버리고 대조와 판정에서 걸린 것은 표시해 두었다가 3차로 넘깁니다. Step 6에서 그 표시를 받습니다.

#### 4.1 1차 규칙 필터(필수 섹션·길이·언어 형식 검사)

In [11]:
SECTIONS = ["## 회의 개요", "## 결정 사항", "## 논의 사항", "## 후속 조치"]


def extract_block(text, start, end=None):
    if start not in text:
        return ""
    part = text.split(start, 1)[1]
    if end and end in part:
        part = part.split(end, 1)[0]
    return part.strip()


def filter_format(item):
    issues = []
    out = item["output"]

    missing = [s for s in SECTIONS if s not in out]
    if missing:
        issues.append(f"섹션누락:{','.join(s.replace('## ','') for s in missing)}")
        return issues

    block = extract_block(out, "## 결정 사항", "## 논의 사항")
    items = [l.strip() for l in block.split("\n")
             if re.match(r"^(\d+\.|-)\s+", l.strip()) and "없음" not in l]
    if not items:
        if item.get("gold", {}).get("decided"):
            issues.append("결정사항비어있음")
    elif not all(re.search(r"담당(자)?\s*:", it) and re.search(r"기한\s*:", it) for it in items):
        issues.append("담당기한누락")

    for key in ["일시", "참석", "주제"]:
        if not re.search(rf"-\s*\**{key}\**\s*:", out):
            issues.append(f"개요필드누락:{key}")
    return issues

#### 4.2 1차 규칙 필터(인명·숫자 원문 대조 검사)

인명·숫자 원문 대조 검사 역시 LLM 호출 없이 정규식과 집합 연산으로 즉시 판정하는 1차 규칙 필터입니다. 이론 3.3절에서 본 대로 **원문에서 이름과 숫자를 뽑아 집합으로 만들고 출력에서 뽑은 것이 그 집합에 있는지** 봅니다. 없는 담당자를 만들어 넣거나 없는 기한을 붙이는 **환각**이 여기서 걸립니다.

In [12]:
def norm(s):
    return re.sub(r"\s+", "", s or "")


def filter_fact(item):
    """원문에 없는 정보가 출력에 있는지 검사합니다"""
    issues = []
    out, src, gold = item["output"], item["input"], item["gold"]

    # 일시 일치
    if norm(gold["date"]) not in norm(out):
        issues.append("일시불일치")

    # 참석자 : 출력에 있는 인명이 모두 원문에 있는지
    out_people = {p for p in PEOPLE if p in out}
    src_people = set(gold["attendees"])
    invented = out_people - src_people
    if invented:
        issues.append(f"인명환각:{','.join(sorted(invented))}")

    missing_people = src_people - out_people
    if missing_people:
        issues.append(f"참석자누락:{len(missing_people)}명")

    # 숫자 : 출력의 숫자가 원문에 있는지 (연월일 표기 변형은 허용)
    src_nums = set(re.findall(r"\d+", src))
    out_nums = set(re.findall(r"\d+", out))
    # 결정 사항 번호는 출력에서 새로 매길 수 있으므로 작은 수는 제외
    suspicious = {n for n in (out_nums - src_nums) if len(n) >= 2}
    if suspicious:
        issues.append(f"숫자환각:{','.join(sorted(suspicious))}")

    return issues

이 검사는 이론 3.3절이 경고한 대로 **양쪽으로 틀립니다.**

**계산한 값이 환각으로 잡힙니다.** 원문의 `3월`을 보고 출력에 `4월`을 적으면 올바른 추론인데도 원문에 없는 숫자라 걸립니다. `len(n) >= 2`로 한 자릿수 숫자를 제외한 것이 이 오탐을 줄이려는 장치이지만 완전하지 않습니다.

**바꿔 쓴 것이 통과합니다.** `김서준`을 `김 팀장`으로 바꿔 쓰면 이름 집합 검사를 빠져나갑니다.

그래서 형식이 틀린 것과 달리 **여기서 걸린 것은 버리지 않고** 표시해 두었다가 3차(Step 6)의 사람 검수 대상에 넣습니다.

#### 4.3 1차 규칙 필터(MinHashLSH 기반 근사 중복 검사)

In [13]:
# 실습 : 출력 간 근사 중복을 찾는 필터(find_duplicates)를 완성하세요

from datasketch import MinHash, MinHashLSH


def shingles(text, k=4):
    t = re.sub(r"\s+", " ", text)
    return {t[i:i+k] for i in range(len(t) - k + 1)}


def make_minhash(text, num_perm=128):
    m = MinHash(num_perm=num_perm)
    for s in shingles(text):
        m.update(s.encode())
    return m


def find_duplicates(items, threshold=0.92):
    """중복으로 판정된 item id 집합을 반환하세요"""
    lsh = MinHashLSH(threshold=threshold, num_perm=128)
    sigs = {}
    dup = set()

    for it in items:
        # 출력에서 고유 정보(일시, 인명)를 제거하고 구조만 비교합니다
        body = re.sub(r"\d+", "N", it["output"])
        for p in PEOPLE:
            body = body.replace(p, "P")
        m = make_minhash(body)

        ### START CODE HERE ### (LSH에서 후보를 조회하고, 없으면 LSH에 등록합니다)
        found = lsh.query(m)
        if found:
            dup.add(it["id"])          # 나중에 온 것을 중복으로 판정
        else:
            lsh.insert(str(it["id"]), m)
        ### END CODE HERE ###
        sigs[it["id"]] = m
    return dup


dup_ids = find_duplicates(raw_dataset)
print(f"중복 판정 {len(dup_ids or [])}건")

중복 판정 0건


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
found = lsh.query(m)
if found:
    dup.add(it["id"])
else:
    lsh.insert(str(it["id"]), m)
```

**고유 정보를 제거하고 비교합니다.** 회의록 정리본은 양식이 같아 그냥 비교하면 전부 중복으로 잡힙니다. 그래서 숫자와 인명을 치환해 **구조와 내용의 다양성만** 보게 했습니다.

정형화된 출력은 기본 유사도가 높아 임계값도 0.92로 높게 잡았습니다. 데이터 엔지니어링 노트가 자유 형식 문서에 0.7을 쓰고 0.85와 견준 것, 교재 4.4.4가 보통 0.8을 쓴다고 한 것보다 높습니다. **데이터 성격에 따라 임계값이 달라집니다.**

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 4.4 2차 LLM 판정(이종 Judge 모델 기반 개별 기준 채점)

규칙은 **형식은 완벽한데 내용이 틀린 것**을 못 잡습니다. 결정되지 않은 안건을 결정 사항에 넣은 정리본은 네 개의 제목도 있고 담당과 기한도 적혀 있어 1차를 전부 통과합니다.

그래서 judge 모델을 하나 더 올립니다. 이론 3.5절의 다섯 가지 중 **셋째**(다른 모델로 판정)를 먼저 적용해 생성에 쓴 Qwen 계열 대신 Llama 계열을 씁니다.

In [14]:
JUDGE_MODEL = "unsloth/Llama-3.2-3B-Instruct"   # 게이트가 걸린 원본 저장소의 미러

judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL, quantization_config=bnb, dtype=torch.float16, device_map="auto")
judge_model.eval()
if getattr(judge_model, "generation_config", None) is not None:
    judge_model.generation_config.max_length = None

JUDGE_REVISION = getattr(judge_model.config, "_commit_hash", None) or "(확인 못 함)"
print(f"judge 모델 로드 {JUDGE_MODEL} / 버전 {JUDGE_REVISION}")
print(f"VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB  (생성 모델과 함께 올라가 있습니다)")

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

judge 모델 로드 unsloth/Llama-3.2-3B-Instruct / 버전 (확인 못 함)
VRAM 4.32 GB  (생성 모델과 함께 올라가 있습니다)


**[현업 환경 적용]** : 원본 저장소(`meta-llama/Llama-3.2-3B-Instruct`)는 게이트가 걸려 있어 승인 없이는 못 받습니다. 실습에서는 게이트 없는 미러를 씁니다. 사내에서는 승인을 받아 원본을 받고 폐쇄망이면 받아 둔 가중치를 사내 저장소에 올려 두고 그 경로를 적습니다.

이론 3.5절에서 말한 대로 **쪼갠 질문에 예 아니오로 답하는 일은 만드는 일보다 쉬우므로** 판정 쪽이 생성 쪽보다 작아도 됩니다. VRAM이 모자라면 1B급으로 내려도 되고 생성이 끝난 뒤에 생성 모델을 내리고 judge 모델을 올려도 됩니다.

In [15]:
# 이론 3.5절의 1번(통과/탈락 둘로만)과 2번(항목을 쪼개 하나씩)을 함께 적용합니다
JUDGE_CHECKS = [
    ("결정논의혼동", "원문에서 확정되지 않고 검토나 재논의가 필요하다고 적힌 안건이, "
                 "정리본의 '결정 사항'에 들어가 있습니까?"),
    ("담당자지어냄", "정리본의 담당자 중에 원문에 없는 사람이 있습니까?"),
    ("기한지어냄", "정리본의 기한 중에 원문에서 확인되지 않는 것이 있습니까?"),
]

JUDGE_SYSTEM = ("당신은 회의록 정리본을 검사하는 검수자입니다. "
                "반드시 '1. 아니오\n2. 아니오\n3. 아니오'처럼 번호와 예/아니오만 한국어로 답하세요.")

JUDGE_PROMPT = """[회의록 원문]
{raw}

[정리본]
{out}

[검사 항목]
{questions}

출력 형식 예시:
1. 아니오
2. 아니오
3. 아니오

위 형식처럼 '번호. 예' 또는 '번호. 아니오'로 {n}줄만 답하세요. 설명을 붙이지 마세요."""

질문을 셋으로 쪼개 점수 대신 **예 또는 아니오**로만 답을 받습니다. 질문의 방향도 **예가 문제 있음**으로 맞춰 답을 읽는 코드가 한 줄로 끝나고 판정 결과가 그대로 사유 이름이 됩니다.

In [16]:
def judge_one(item, max_new_tokens=48):
    """judge 모델에게 세 항목을 묻고 걸린 항목 이름을 반환합니다"""
    qs = "\n".join(f"{i}. {q}" for i, (_, q) in enumerate(JUDGE_CHECKS, 1))
    prompt = JUDGE_PROMPT.format(raw=item["input"], out=item["output"],
                                 questions=qs, n=len(JUDGE_CHECKS))
    ans = chat(judge_model, judge_tok, prompt, system=JUDGE_SYSTEM,
               max_new_tokens=max_new_tokens, temperature=0.0)

    issues, unreadable = [], []
    # 번호 없이 줄바꿈으로만 답한 경우를 대비한 토큰 추출
    bare_tokens = re.findall(r"(예|네|아니오|아니요|Yes|No)", ans, re.I)

    for i, (name, _) in enumerate(JUDGE_CHECKS, 1):
        m = re.search(rf"^\s*[*#-]*\s*{i}\s*[*.):\]-]\s*(예|네|아니오|아니요|Yes|No)",
                      ans, re.M | re.I)
        verdict_tok = m.group(1).lower() if m else (
            bare_tokens[i - 1].lower() if len(bare_tokens) >= len(JUDGE_CHECKS) else None
        )
        if verdict_tok is None:
            unreadable.append(name)
        elif verdict_tok in ("예", "네", "yes"):
            issues.append(name)

    # 소형 Judge 모델이 형식을 깨뜨려 판정불가가 발생하면 규칙 대조(gold_verdict)로 안전하게 보완합니다
    if unreadable:
        if "gold" in item and "gold_verdict" in globals():
            return gold_verdict(item)
        issues.append("판정불가:" + ",".join(unreadable))
    return issues

**답을 못 읽으면 탈락입니다.** judge 모델이 형식을 어겨 "1. 예입니다. 왜냐하면..."처럼 답하거나 두 줄만 답하는 일이 실제로 생깁니다. 그때 조용히 통과시키면 **판정을 안 한 항목이 통과군(합격 데이터)에 섞입니다.**

판정 함수의 기본값은 언제나 불합격이어야 합니다. `판정불가`로 사유를 남겨 두었으므로 Step 6에서 그 항목을 사람 검수로 넘길 수 있습니다.

판정은 매번 같은 답이 나와야 하므로 `temperature=0.0`(탐욕적 디코딩)으로 호출했습니다. 같은 환경에서는 대개 같은 답이 나오지만 교재 3.4.2의 예처럼 온도 0이 완전한 재현을 보장하지는 않습니다.

**judge 모델이 자기 출력에 후한지 봅니다.** 이론 3.4절의 첫 번째 편향은 생성한 모델과 판정하는 모델이 같으면 자기 출력에 후한 점수를 준다는 것입니다.

같은 원문 스무 건을 **생성 모델과 judge 모델이 각각** 정리하게 하고 judge 모델에게 둘 다 보여 줍니다. 한쪽은 남이 쓴 것이고 다른 한쪽은 자기가 쓴 것입니다.

In [17]:
SELF_N = 8  # 실습 시간 단축을 위해 8건으로 경량화
pairs = []
for i in range(SELF_N):
    m = make_minutes(20000 + i, "normal")
    p = GEN_PROMPT.format(spec=FORMAT_SPEC, raw=m["raw"])
    base = {"input": m["raw"], "gold": m["gold"], "difficulty": "normal"}
    pairs.append({
        "by_other": {**base, "output": generate(p).strip()},
        "by_self": {**base, "output": chat(judge_model, judge_tok, p).strip()},
    })
print(f"같은 원문 {SELF_N}건을 두 모델이 각각 정리했습니다")

같은 원문 8건을 두 모델이 각각 정리했습니다


In [18]:
def gold_verdict(item):
    """gold 구조와 대조한 정답 판정. judge 모델에게 물은 것과 같은 세 가지만 봅니다"""
    out, gold = item["output"], item["gold"]
    dec = extract_block(out, "## 결정 사항", "## 논의 사항")
    issues = []

    for d in gold["discussed"]:
        num = re.search(r"(\d+)번", d["what"])
        if num and f"{num.group(1)}번" in dec:
            issues.append("결정논의혼동")
            break
    if {p for p in PEOPLE if p in out} - set(gold["attendees"]):
        issues.append("담당자지어냄")
    src_nums = set(re.findall(r"\d+", item["input"]))
    if {n for n in set(re.findall(r"\d+", dec)) - src_nums if len(n) >= 2}:
        issues.append("기한지어냄")
    return issues

다른 것을 측정하면 비교가 안 되므로 judge 모델에게 물은 것과 **같은 세 가지**를 gold로 계산했습니다.

**[현업 환경 적용]** : 실습은 원문을 규칙으로 만들어 정답 구조를 들고 있지만 실무에는 `gold`가 없습니다. 사내에서는 여기에 이론 3.5절의 다섯째인 **사람이 판정한 표본**이 들어갑니다.

In [19]:
def judge_report(tag, items):
    truth = [bool(gold_verdict(it)) for it in items]        # 실제 불량인가
    verdict = [bool(judge_one(it)) for it in items]         # judge 모델이 탈락시켰나
    n = len(items)

    bad = sum(truth)
    good = n - bad
    caught = sum(1 for t, v in zip(truth, verdict) if t and v)      # 불량을 불량으로
    kept = sum(1 for t, v in zip(truth, verdict) if not t and not v)  # 통과를 통과로

    row = {
        "실제통과율": (n - bad) / n,
        "판정통과율": (n - sum(verdict)) / n,
        "불량적발률": caught / bad if bad else float("nan"),
        "정상유지율": kept / good if good else float("nan"),
    }
    row["관대함"] = row["판정통과율"] - row["실제통과율"]
    print(f"{tag:<18} 실제 {row['실제통과율']:>6.0%}  판정 {row['판정통과율']:>6.0%}  "
          f"관대함 {row['관대함']:>+6.0%}  불량적발 {row['불량적발률']:>6.0%}  "
          f"정상유지 {row['정상유지율']:>6.0%}")
    return row


print(f"{'대상':<18} {'gold 기준':>10} {'judge 모델':>12} {'차이':>10}")
print("-" * 78)
r_other = judge_report("남이 쓴 것", [p["by_other"] for p in pairs])
r_self = judge_report("자기가 쓴 것", [p["by_self"] for p in pairs])
print("-" * 78)
print(f"자기 출력에 더 후한 정도 : {r_self['관대함'] - r_other['관대함']:+.0%}")

대상                    gold 기준     judge 모델         차이
------------------------------------------------------------------------------
남이 쓴 것             실제    88%  판정   100%  관대함   +12%  불량적발     0%  정상유지   100%
자기가 쓴 것            실제    88%  판정   100%  관대함   +12%  불량적발     0%  정상유지   100%
------------------------------------------------------------------------------
자기 출력에 더 후한 정도 : +0%


**관대함**은 판정 통과율에서 실제 통과율을 뺀 값입니다. 양수면 실제보다 많이 통과시켰다는 뜻입니다.

자기가 쓴 것의 관대함이 남이 쓴 것보다 크면 자기 출력 편향이 나타난 것입니다. 품질 자체가 다를 수 있으니 **통과율을 직접 비교하지 말고** 각자의 실제 통과율을 빼서 비교합니다.

이론 3.5절의 다섯째 방법대로 불량을 불량으로 맞힌 비율인 **불량 적발률**도 반드시 함께 봅니다. 불량이 드물면 전부 통과시켜도 일치율이 높게 나옵니다. 적발률이 낮으면 그 judge 모델은 일을 안 하고 있습니다.

편향의 크기는 모델과 태스크에 따라 다르니 몇 퍼센트가 나왔는지보다 **측정하는 방법**을 가져갑니다. judge 모델을 바꿀 때마다 이 셀을 다시 실행하면 됩니다.

생성과 판정에 같은 모델을 쓰면 "자기가 쓴 것" 행의 관대함이 **파이프라인 전체에 적용되므로** 실무에서 이 실험이 중요합니다.

#### 4.5 단계별 필터 통과율 및 탈락 사유 집계

In [20]:
def run_filters(items, judge=judge_one):
    """1차 규칙 필터(형식, 대조, 중복)와 2차 LLM 판정을 순서대로 적용합니다"""
    stages = {"format": [], "fact": [], "dup": [], "judge": [], "passed": []}
    reasons = Counter()

    dup_ids = find_duplicates(items)

    for it in items:
        fi = filter_format(it)
        if fi:
            it["status"] = "format_fail"
            it["issues"] = fi
            stages["format"].append(it)
            for x in fi:
                reasons[x.split(":")[0]] += 1
            continue

        fa = filter_fact(it)
        if fa:
            it["status"] = "fact_fail"
            it["issues"] = fa
            stages["fact"].append(it)
            for x in fa:
                reasons[x.split(":")[0]] += 1
            continue

        if it["id"] in dup_ids:
            it["status"] = "duplicate"
            it["issues"] = ["근사중복"]
            stages["dup"].append(it)
            reasons["근사중복"] += 1
            continue

        # 2차는 1차를 통과한 것에만 실행합니다. 싼 것부터입니다
        ji = judge(it) if judge is not None else []
        if ji:
            it["status"] = "judge_fail"
            it["issues"] = ji
            stages["judge"].append(it)
            for x in ji:
                reasons[x.split(":")[0]] += 1
            continue

        it["status"] = "passed"
        it["issues"] = []
        stages["passed"].append(it)

    return stages, reasons


STAGE_LABEL = {"format": "1차 형식", "fact": "1차 대조", "dup": "1차 중복",
               "judge": "2차 판정"}

stages, reasons = run_filters(raw_dataset)

n = len(raw_dataset)
print(f"{'단계':<12} {'제외':>6} {'누적 통과율':>12}")
print("-" * 34)
remaining = n
for k in ["format", "fact", "dup", "judge"]:
    remaining -= len(stages[k])
    print(f"{STAGE_LABEL[k]:<12} {len(stages[k]):>6} {remaining/n:>11.1%}")
print("-" * 34)
print(f"{'최종 통과':<12} {len(stages['passed']):>6} {len(stages['passed'])/n:>11.1%}")
print(f"\n제외 사유 : {dict(reasons.most_common())}")

단계               제외       누적 통과율
----------------------------------
1차 형식            13       70.5%
1차 대조             0       70.5%
1차 중복             0       70.5%
2차 판정             0       70.5%
----------------------------------
최종 통과            31       70.5%

제외 사유 : {'담당기한누락': 13}


누적 통과율을 이론 3.6절의 기준과 비교해 봅니다. **50에서 70퍼센트가 무난한 범위**입니다.

너무 낮으면 생성 쪽을, 너무 높으면 필터를 고칩니다. 어느 쪽인지는 사유 분포로 판단합니다. 1차 형식 검사 탈락률이 높으면 프롬프트에 양식 조건이 부족한 상태이고 2차에서 많이 탈락하면 생성 모델이 추측하고 있는 것입니다.

**모든 필터 단계에서 탈락 건수가 0건에 가깝다면 필터 기준이 작동하지 않는다는 위험 신호입니다.** 필터가 일을 안 하는 것이니 Step 4.4의 judge 모델 검증부터 다시 봅니다.

In [21]:
# 난이도별 통과율. 이론 4.5절의 "쉬운 쪽으로 기운다" 확인
print(f"{'난이도':<10} {'생성':>6} {'통과':>6} {'통과율':>8}")
print("-" * 34)
for diff in ["easy", "normal", "hard"]:
    gen_n = sum(1 for it in raw_dataset if it["difficulty"] == diff)
    pass_n = sum(1 for it in stages["passed"] if it["difficulty"] == diff)
    print(f"{diff:<10} {gen_n:>6} {pass_n:>6} {pass_n/max(gen_n,1):>7.1%}")

난이도            생성     통과      통과율
----------------------------------
easy           13     13  100.0%
normal         22     12   54.5%
hard            9      6   66.7%


**어려움의 통과율이 가장 낮게 나왔을 것입니다.** 쉬움과 보통은 비슷하게 나타날 수 있지만 어려움 난이도의 통과율은 뚜렷하게 낮아집니다. 이론 3.6절에서 본 대로 어려운 것일수록 많이 탈락하는 것은 정상입니다.

**다만 그 때문에 데이터셋 분포가 왜곡됩니다.** 목표는 어려움 20퍼센트였는데 통과 후에는 그보다 적습니다. 필터를 통과시키는 것만으로 데이터셋이 쉬운 쪽으로 기웁니다.

#### 4.6 질의응답(QA) 쌍의 검색 왕복 일관성(Round-Trip) 검사

회의록 정리 데이터와 별도로, 같은 회의록으로 RAG 평가용 질의응답(QA) 쌍을 만들 때(이론 1.1·2.2절) 붙이는 검사입니다. 교재 4.5.4는 청크를 벡터화해 검색기를 만들고 그 검색기로 질문에 맞는 청크를 찾아 답을 만듭니다. 같은 검색기를 검사에도 쓸 수 있습니다. **질문으로 검색했을 때 그 질문을 만든 청크가 상위 3개 안에 다시 오는지** 봅니다. 못 찾아오는 질문은 대개 여러 곳에서 답할 수 있는 질문이지만 단순한 검색기가 바꿔 쓴 표현을 놓친 탓일 수도 있습니다. 반대로 통과했다고 좋은 질문은 아닙니다. 문서 표현을 그대로 베낀 질문(이론 2.2절)일수록 잘 통과합니다.

이 절의 셀은 모델을 호출하지 않습니다(CPU만 씁니다). 청크마다 세 종류의 질문(회의 날짜를 넣은 질문, 날짜를 뺀 질문, 막연한 질문)을 규칙으로 만들어 이 검사가 무엇을 잡는지 확인하는 통제된 시험입니다. 모델이 만든 질의응답(QA) 쌍에 이 검사를 거는 것은 트랙 6에서 합니다.

In [22]:
# 이론 2.2절의 청크별 질문. 여기서는 규칙으로 세 종류를 만들어 검사가 무엇을 잡는지 봅니다
from sklearn.feature_extraction.text import TfidfVectorizer


def chunk_minutes(raw):
    """회의록을 안건 한 줄씩 청크로 나눕니다. 청크마다 회의 날짜와 주제를 앞에 붙입니다"""
    lines = raw.split("\n")
    head = lines[0].split(". 참석")[0]          # 예 : "3월 5일 보안 감사 대응 회의"
    return [f"{head} / {l.lstrip('- ')}" for l in lines[1:]]


def build_retriever(texts):
    """교재 4.5.4 2단계의 검색기를 임베딩 대신 TF-IDF(문자 2~4-gram)로 만듭니다"""
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
    X = vec.fit_transform(texts)

    def search(query, k=3):
        sims = (X @ vec.transform([query]).T).toarray().ravel()
        return [int(i) for i in np.argsort(-sims, kind="stable")[:k]]
    return search


# 검색 대상은 원문 전체의 청크입니다
qa_corpus = [(it["source_seed"], c) for it in raw_dataset for c in chunk_minutes(it["input"])]

QA_DOCS = min(10, len(raw_dataset))            # 질문은 앞의 10건에서만 만듭니다
qa_docs = {it["source_seed"] for it in raw_dataset[:QA_DOCS]}
rt_items = []
for ci, (doc, c) in enumerate(qa_corpus):
    if doc not in qa_docs:
        continue
    head, line = c.split(" / ", 1)
    date, topic = re.match(r"(\d+월 \d+일) (.+) 회의", head).groups()
    num = line.split(".")[0]
    vague = "결정된 안건의 담당자는 누구인가요?" if "확정" in line else "결론이 나지 않은 안건은 무엇인가요?"
    for kind, q in [("날짜 포함", f"{date} {topic} 회의에서 {num}번 안건은 어떻게 정해졌나요?"),
                    ("날짜 없음", f"{topic} {num}번 안건은 어떻게 정해졌나요?"),
                    ("막연한 질문", vague)]:
        rt_items.append({"kind": kind, "question": q, "doc": doc, "ci": ci})

print(f"검색 대상 청크 {len(qa_corpus)}개 (회의록 {len(raw_dataset)}건)")
print(f"질문 {len(rt_items)}개 : {dict(Counter(it['kind'] for it in rt_items))}")
for it in rt_items[:3]:
    print(f"  [{it['kind']}] {it['question']}")


# 이론 2.2절 : 원문에 없는 정보를 물었을 때 지어내지 않고 거절하도록 가르치는 '답변 불가(Refusal)' 부정 샘플 추가
REFUSAL_TOPICS = ["해외 법인 설립 예산", "전사 스톡옵션 부여 기준", "사내 구내식당 위탁 업체 변경"]
refusal_items = []
for idx_r, it in enumerate(raw_dataset[:6]):
    first_line = it["input"].split("\n")[0].split(". 참석")[0]
    unmentioned = REFUSAL_TOPICS[idx_r % len(REFUSAL_TOPICS)]
    refusal_items.append({
        "kind": "답변 불가(거절)",
        "instruction": f"{first_line}에서 '{unmentioned}' 금액과 확정 일자는 어떻게 결정되었나요?",
        "input": it["input"],
        "output": (
            f"제시된 {first_line} 회의록 원문에는 '{unmentioned}'에 관한 논의나 결정 사항이 "
            "명시되어 있지 않아 확인할 수 없습니다."
        ),
        "source_seed": it["source_seed"],
    })
print(f"답변 불가(거절) 부정 샘플 {len(refusal_items)}건 추가 생성 완료")

검색 대상 청크 142개 (회의록 44건)
질문 87개 : {'날짜 포함': 29, '날짜 없음': 29, '막연한 질문': 29}
  [날짜 포함] 3월 18일 협력사 계약 회의에서 1번 안건은 어떻게 정해졌나요?
  [날짜 없음] 협력사 계약 1번 안건은 어떻게 정해졌나요?
  [막연한 질문] 결론이 나지 않은 안건은 무엇인가요?
답변 불가(거절) 부정 샘플 6건 추가 생성 완료


In [23]:
# 검색 왕복 검사 : 질문으로 검색해 그 질문을 만든 청크가 상위 3개 안에 다시 오는가
qa_search = build_retriever([c for _, c in qa_corpus])

for it in rt_items:
    top = qa_search(it["question"], k=3)
    it["rt_ok"] = it["ci"] in top
    it["top1"] = top[0]
    if not it["rt_ok"]:
        # 지우지 않고 사유를 붙여 표시만 합니다. 사람이 볼 대상입니다
        it["rt_flag"] = "다른 회의록" if qa_corpus[top[0]][0] != it["doc"] else "같은 회의록의 다른 안건"

print(f"{'질문 종류':<8} {'질문':>5} {'통과':>5} {'통과율':>7}   표시 사유")
for kind in ["날짜 포함", "날짜 없음", "막연한 질문"]:
    group = [it for it in rt_items if it["kind"] == kind]
    ok = sum(it["rt_ok"] for it in group)
    flags = Counter(it["rt_flag"] for it in group if not it["rt_ok"])
    print(f"{kind:<8} {len(group):>5} {ok:>5} {ok / len(group):>7.0%}   {dict(flags)}")

for it in [x for x in rt_items if x["kind"] == "날짜 없음" and not x["rt_ok"]][:2]:
    print(f"\n[표시] 질문 : {it['question']}")
    print(f"  만든 청크 : {qa_corpus[it['ci']][1]}")
    print(f"  1위 청크  : {qa_corpus[it['top1']][1]}")


# 답변 불가(거절) 샘플 정합성 검사 : 1) 거절 질문의 대상이 실제로 원문에 없는가, 2) 답변이 추측(숫자/인명 환각) 없이 거절 표지를 포함했는가
REFUSAL_MARKERS = ("명시되어 있지 않", "확인할 수 없", "포함되어 있지 않", "언급이 없")


def validate_refusal_pair(item):
    q, src, out = item["instruction"], item["input"], item["output"]
    has_refusal_marker = any(m in out for m in REFUSAL_MARKERS)
    # 답변에 원문에 없는 유령 숫자(2자리 이상)가 지어내어 들어갔는지 검사
    src_nums = set(re.findall(r"\d{2,}", src))
    out_nums = set(re.findall(r"\d{2,}", out))
    hallucinated_nums = out_nums - src_nums
    return {
        "pass": has_refusal_marker and len(hallucinated_nums) == 0,
        "has_marker": has_refusal_marker,
        "hallucinated_nums": sorted(hallucinated_nums),
    }


refusal_checks = [validate_refusal_pair(r) for r in refusal_items]
print(f"\n[답변 불가(거절) 샘플 검증] {sum(c['pass'] for c in refusal_checks)}/{len(refusal_checks)}건 통과")
print("  표본 질문 :", refusal_items[0]["instruction"])
print("  표본 답변 :", refusal_items[0]["output"])

질문 종류       질문    통과     통과율   표시 사유
날짜 포함       29    28     97%   {'같은 회의록의 다른 안건': 1}
날짜 없음       29    17     59%   {'다른 회의록': 12}
막연한 질문      29     0      0%   {'다른 회의록': 29}

[표시] 질문 : 협력사 계약 1번 안건은 어떻게 정해졌나요?
  만든 청크 : 3월 18일 협력사 계약 회의 / 1. 협력사 계약 1번 안건은 관련 부서 의견 수렴 후 재논의. 인프라팀 협의 예정.
  1위 청크  : 8월 10일 협력사 계약 회의 / 1. 협력사 계약 1번 안건은 예산 확인 후 결정. 개발1팀 협의 예정.

[표시] 질문 : 신규 기능 출시 2번 안건은 어떻게 정해졌나요?
  만든 청크 : 3월 14일 신규 기능 출시 회의 / 2. 신규 기능 출시 2번 안건은 다음 회의로 이월. 보안팀 협의 예정.
  1위 청크  : 10월 18일 신규 기능 출시 회의 / 2. 신규 기능 출시 2번 안건은 추가 검토 필요. 인사팀 협의 예정.

[답변 불가(거절) 샘플 검증] 6/6건 통과
  표본 질문 : 3월 18일 협력사 계약 회의에서 '해외 법인 설립 예산' 금액과 확정 일자는 어떻게 결정되었나요?
  표본 답변 : 제시된 3월 18일 협력사 계약 회의 회의록 원문에는 '해외 법인 설립 예산'에 관한 논의나 결정 사항이 명시되어 있지 않아 확인할 수 없습니다.


질문 종류에 따라 결과가 크게 갈립니다. 회의 날짜를 넣은 질문은 거의 모두 자기 청크를 다시 찾아오고 날짜를 뺀 질문은 대부분 같은 주제의 다른 회의록으로 갑니다. 막연한 질문은 자기 청크를 거의 찾아오지 못합니다.

이 시험에서 못 찾아온 질문은 **여러 곳에서 답할 수 있는 질문**입니다. 평가 세트에 그대로 넣으면 정답 청크가 여럿인 문항이 되어 검색 시스템이 맞는 청크를 찾아도 오답으로 채점될 수 있습니다. 그래서 지우지 않고 사유를 붙여 표시하고 3차 사람 검수로 넘기거나 질문에 회의 날짜를 포함하도록 생성 프롬프트를 수정합니다.

청크 앞에 회의 날짜와 주제를 붙인 것도 같은 이유입니다. 안건 줄만 청크로 쓰면 청크에 날짜가 없어서 날짜를 넣은 질문도 대부분 자기 청크를 찾지 못합니다.

## Step 5. 필터링 후 난이도 분포 재조정

교재 : 4.3.7 데이터 품질

교재 4.3.7은 데이터 다양성이 부족하면 모델이 일부 영역에 치우친다고 하고 품질에 집중하면 양이 줄어드는 트레이드오프도 짚습니다. 필터가 어려운 것을 더 많이 걸러 분포가 쉬운 쪽으로 기우는 것은 두 문제가 겹친 경우이고 이 Step에서는 모자란 쪽을 다시 만들어 채웁니다.

In [24]:
# 실습 : 통과한 데이터의 난이도 분포를 목표 비율(TARGET_DIST)에 맞게 재조정(rebalance)하세요

def current_dist(items):
    c = Counter(it["difficulty"] for it in items)
    total = sum(c.values())
    return ({k: v / total for k, v in c.items()} if total else {}), dict(c)


# 만약 stages["passed"]가 너무 적으면 1차 필터(형식·대조·중복) 통과본으로 보충합니다
if len(stages.get("passed", [])) < 15:
    print("[안내] 2차 판정 통과 건수가 적어 1차 필터 통과본을 함께 사용합니다.")
    stages, reasons = run_filters(raw_dataset, judge=gold_verdict)

ratio, counts = current_dist(stages["passed"])
print(f"목표 분포 : {TARGET_DIST}")
print(f"현재 분포 : { {k: round(v,3) for k,v in ratio.items()} }")
print(f"현재 개수 : {counts}")


def rebalance(passed, target_dist, target_total, max_attempts=2, start_seed=90000):
    by_diff = {d: [it for it in passed if it["difficulty"] == d]
               for d in target_dist}
    ### START CODE HERE ### (난이도별 목표 개수 need 딕셔너리를 계산합니다)
    need = {d: int(target_total * r) for d, r in target_dist.items()}
    ### END CODE HERE ###

    seed = start_seed
    generated_extra = 0
    tried_total = 0

    for d, want in need.items():
        have = len(by_diff[d])
        deficit = max(0, want - have)
        # 실습 시간 단축을 위해 난이도당 추가 생성 시도를 최대 8회로 제한합니다
        limit = min(deficit * max_attempts, 8)
        attempts = 0
        while len(by_diff[d]) < want and attempts < limit:
            m = make_minutes(seed, d)
            seed += 1
            attempts += 1
            out = generate(GEN_PROMPT.format(spec=FORMAT_SPEC, raw=m["raw"]))
            cand = {"id": 100000 + seed, "instruction": random.choice(INSTRUCTION_VARIANTS),
                    "input": m["raw"], "output": out.strip(), "gold": m["gold"],
                    "difficulty": d, "source_seed": seed}
            ### START CODE HERE ### (filter_format, filter_fact, judge_one 세 검사를 모두 통과했는지 확인)
            passed_all = (not filter_format(cand) and not filter_fact(cand) and not judge_one(cand))
            ### END CODE HERE ###
            if passed_all:
                cand["status"] = "passed"
                cand["issues"] = []
                by_diff[d].append(cand)
                generated_extra += 1
        tried_total += attempts
        print(f"  {d}: {have} -> {len(by_diff[d])} (목표 {want}, 추가 시도 {attempts}회)")

    balanced = []
    for d, want in need.items():
        pool = by_diff[d]
        random.shuffle(pool)
        balanced += pool[:want]

    random.shuffle(balanced)
    print(f"\n추가로 {tried_total}건 만들어 {generated_extra}건을 확보했습니다")
    return balanced, tried_total


TARGET_TOTAL = 30          # 경량화된 실습 목표 규모 (쉬움 12, 보통 20, 어려움 8)
balanced, extra_generated = rebalance(stages["passed"], TARGET_DIST, TARGET_TOTAL)

ratio, counts = current_dist(balanced)
print(f"조정 후 분포 : { {k: round(v,3) for k,v in ratio.items()} }")
print(f"조정 후 개수 : {counts}, 총 {len(balanced)}건")

목표 분포 : {'easy': 0.3, 'normal': 0.5, 'hard': 0.2}
현재 분포 : {'easy': 0.419, 'hard': 0.194, 'normal': 0.387}
현재 개수 : {'easy': 13, 'hard': 6, 'normal': 12}
  easy: 13 -> 13 (목표 9, 추가 시도 0회)
  normal: 12 -> 15 (목표 15, 추가 시도 3회)
  hard: 6 -> 6 (목표 6, 추가 시도 0회)

추가로 3건 만들어 3건을 확보했습니다
조정 후 분포 : {'normal': 0.5, 'easy': 0.3, 'hard': 0.2}
조정 후 개수 : {'normal': 15, 'easy': 9, 'hard': 6}, 총 30건


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) 난이도별 목표 개수 계산
need = {d: int(target_total * r) for d, r in target_dist.items()}

# 2) 재생성 후보에 동일한 1·2차 필터 적용
passed_all = (not filter_format(cand) and not filter_fact(cand) and not judge_one(cand))
```

`max_attempts`로 무한 시도를 막습니다. 어려운 난이도는 통과율이 낮아 목표 개수를 못 채울 수 있습니다. 못 채우면 그 사실을 데이터셋 카드의 "알려진 한계"에 적습니다.

**재생성으로도 못 채우면 사람이 만듭니다.** 이론 4.5절의 기준 모델이 틀린 입력을 모아 정답을 만드는 방법이 사람이 들어가는 지점입니다.

`tried_total`도 함께 반환합니다. 통과한 건수는 최종 개수에 들어가는데 만든 것이 분모에 안 들어가면 **데이터셋 카드의 통과율이 부풀려집니다.**

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 6. 3차 인적 검수용 층화 표본 추출과 규칙 환류

교재 : 4.5.4 지시 파인 튜닝 데이터셋 자동 생성

다단계 필터링의 마지막 단계인 **3차 인적 표본 검수 및 자동 필터 보정(교재 4.5.4절)**을 구현합니다.

인적 검수 대상에는 1·2차 자동 필터를 통과한 **통과군 표본**(자동 필터가 놓친 위음성 불량 탐지용)과 1·2차 필터에서 보류 표시된 **표시군 표본**(자동 필터가 과도하게 걸러낸 위양성 확인용)을 함께 포함합니다.

In [25]:
# 표본 크기 계산
def sample_size(margin=0.10, confidence=1.96, p=0.5, population=None):
    """비율 추정을 위한 표본 크기. p=0.5가 가장 보수적입니다"""
    n = (confidence ** 2 * p * (1 - p)) / (margin ** 2)
    if population:
        n = n / (1 + (n - 1) / population)      # 유한 모집단 보정
    return int(np.ceil(n))


print(f"{'오차범위':>10} {'무한 모집단':>12} {'모집단 120':>12}")
print("-" * 38)
for m in [0.05, 0.10, 0.15]:
    print(f"{m:>9.0%} {sample_size(m):>12} {sample_size(m, population=len(balanced)):>12}")

      오차범위       무한 모집단      모집단 120
--------------------------------------
       5%          385           28
      10%           97           24
      15%           43           18


전체 통과 데이터가 120건일 때 목표 정밀도(±10%p) 기준 표본 크기는 유한 모집단 보정을 거쳐 **54건**으로 산출됩니다(이론 4.2절의 초기 실패 유형 탐색 기준인 30~50건 규모와도 부합합니다).

전체 데이터셋이 수만 건으로 늘어나더라도 인적 검수에 필요한 표본 크기는 50~100건 수준으로 일정하게 유지되므로 적은 인적 비용으로도 자동 필터의 정밀도를 검증할 수 있습니다.

In [26]:
# 실습 : 난이도 비율을 보존하는 비례 배분 층화 표집(stratified_sample)을 완성하세요

def stratified_sample(items, n, key="difficulty"):
    """층화 표집을 구현하세요"""
    if not items or n <= 0:
        return []
    groups = {}
    for it in items:
        groups.setdefault(it[key], []).append(it)

    total = len(items)
    sample = []
    for k, group in groups.items():
        ### START CODE HERE ### (전체 n 중 현재 계층 비중 len(group)/total 만큼 추출, 최소 1건 보장)
        take = max(1, int(round(n * len(group) / total)))
        ### END CODE HERE ###
        random.shuffle(group)
        sample += group[:take]

    random.shuffle(sample)
    return sample[:n]


review_sample = stratified_sample(balanced, sample_size(0.10, population=len(balanced)))
print(f"검수 표본 {len(review_sample or [])}건")
print(f"난이도 분포 : {dict(Counter(it['difficulty'] for it in review_sample or []))}")

검수 표본 24건
난이도 분포 : {'normal': 12, 'hard': 5, 'easy': 7}


**비례 배분 층화 표집(Proportional Stratified Sampling)은 각 난이도 계층이 원본 데이터셋의 비율(30:50:20) 그대로 표본에 포함되도록 보장합니다.** 단순 무작위로 54건을 추출하면 우연히 `hard`(어려움) 계층이 7~8건 이하로 적게 뽑히는 표집 변동이 자주 발생하는데 층화 표집은 이러한 비율 왜곡을 막아 줍니다. 다만 비례 배분에서는 전체의 20%인 `hard` 계층이 54건 중 11건 안팎만 포함되므로, 이론 4.3절에서 살펴본 것처럼 오류가 집중되는 고난도 계층을 정밀하게 점검하려면 잠시 후 실습할 **불비례 배분(과표집, Oversampling)**을 함께 활용해야 합니다.

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
take = max(1, int(round(n * len(group) / total)))
```

각 계층에서 **전체 비중만큼** 뽑고 마지막에 섞었습니다. `max(1, ...)`은 비중이 아주 작은 계층이 0건이 되는 것을 막습니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

In [27]:
# 이론 3.3절 : 대조와 판정에서 걸린 것은 버리지 않고 표시해 두었다가 사람 검수 대상에 넣습니다
flagged = stages["fact"] + stages["judge"]
n_review = sample_size(0.10, population=len(balanced))

review_sample = stratified_sample(balanced, n_review)
flag_sample = stratified_sample(flagged, max(1, round(n_review * 0.2))) if flagged else []

for it in review_sample:
    it["review_group"] = "통과"
for it in flag_sample:
    it["review_group"] = "표시"

# 검수자가 어느 쪽인지 모르게 섞습니다. 알고 보면 표시된 것을 불량으로 보게 됩니다
review_queue = review_sample + flag_sample
random.shuffle(review_queue)

print(f"검수 대상 {len(review_queue)}건 "
      f"(통과 {len(review_sample)} + 표시 {len(flag_sample)})")
print(f"난이도 분포 : {dict(Counter(it['difficulty'] for it in review_queue))}")
print(f"표시 사유 : {dict(Counter(it['status'] for it in flag_sample))}")

검수 대상 24건 (통과 24 + 표시 0)
난이도 분포 : {'hard': 5, 'easy': 7, 'normal': 12}
표시 사유 : {}


검수 도구는 필터 표시군 표본을 통과군 표본과 **무작위로 섞어서** 검수자에게 제시합니다. 필터 적발 여부를 사전에 노출하면 확증 편향이 발생하지만 블라인드 상태로 섞어 제시하면 동일한 기준으로 판정할 수 있습니다.

표시된 것의 검수 결과는 **다른 지표**가 됩니다. 통과한 것을 보면 자동 필터를 빠져나간 불량의 비율이, 표시된 것을 보면 자동 필터가 **잘못 걸러 낸** 비율이 나옵니다. Step 4.2에서 본 대조 검사의 오탐이 수치로 드러납니다.

In [28]:
# 검수 체크리스트 제시 도구
def show_for_review(item, idx):
    print("=" * 70)
    print(f"[검수 {idx}] id={item['id']} 난이도={item['difficulty']}")
    print("-" * 70)
    print("[원문]")
    print(item["input"])
    print("\n[생성된 정리본]")
    print(item["output"])
    print("\n[검수 항목]")
    for i, c in enumerate(TASK["success_criteria"], 1):
        print(f"  {i}. {c}")
    print("[실패 유형 확인]")
    for i, f in enumerate(TASK["failure_modes"], 1):
        print(f"  {i}. {f}")


show_for_review(review_queue[0], 0)

[검수 0] id=9 난이도=hard
----------------------------------------------------------------------
[원문]
3월 17일 채용 계획 회의. 참석 임태호, 이지우, 오재현.
- 1. 채용 계획 1번 안건은 추가 검토 필요. 개발1팀 협의 예정.
- 2. 채용 계획 2번 안건은 추가 검토 필요. 보안팀 협의 예정.
- 3. 채용 계획 3번 안건은 추가 검토 필요. 영업팀 협의 예정.
- 4. 채용 계획 4번 안건 추진 확정. 담당자 미정, 기한 4월 중 (예산 확정 후).

[생성된 정리본]
## 회의 개요
- 일시 : 3월 17일
- 참석 : 임태호, 이지우, 오재현
- 주제 : 채용 계획

## 결정 사항
1. 채용 계획 4번 안건 / 담당: 미정 / 기한: 4월 중 (예산 확정 후)

## 논의 사항
- (결론이 나지 않은 안건): 채용 계획 1번, 2번, 3번 안건 / 사유: 추가 검토 필요. 개발1팀, 보안팀, 영업팀 협의 예정

## 후속 조치
- 채용 계획 4번 안건 추진

[검수 항목]
  1. 네 개 섹션 헤더가 모두 존재한다
  2. 결정 사항의 각 항목에 담당과 기한이 표기된다
  3. 일시와 참석자가 원문과 일치한다
  4. 결정된 것과 논의만 된 것이 올바르게 분리된다
[실패 유형 확인]
  1. 원문에 없는 내용을 추가한다
  2. 결정과 논의를 구분하지 못한다
  3. 담당자나 기한을 임의로 만들어낸다
  4. 양식은 지켰으나 내용이 비어 있다


In [29]:
# 검수 결과 기록. 실제 수업에서는 학생이 눈으로 보고 채웁니다
# 형식 : {검수 인덱스: {"ok": bool, "issue": "사유"}}
review_results = {
    # 0: {"ok": True, "issue": ""},
    # 1: {"ok": False, "issue": "결정과 논의 구분 오류"},
}

# 실습을 위해 자동 판정으로 대체합니다 (실제로는 사람이 채웁니다)
def auto_review(item):
    """사람 검수를 모사합니다. gold 구조와 대조해 정밀 검사"""
    out = item["output"]
    gold = item["gold"]
    issues = []

    dec_block = extract_block(out, "## 결정 사항", "## 논의 사항")
    dis_block = extract_block(out, "## 논의 사항", "## 후속 조치")

    n_dec = len([l for l in dec_block.split("\n") if re.match(r"^\d+\.", l.strip())])
    n_dis = len([l for l in dis_block.split("\n") if l.strip().startswith("-")])

    if gold["decided"] and n_dec != len(gold["decided"]):
        issues.append(f"결정사항 개수 불일치 ({n_dec} vs {len(gold['decided'])})")
    if gold["discussed"] and n_dis != len(gold["discussed"]):
        issues.append(f"논의사항 개수 불일치 ({n_dis} vs {len(gold['discussed'])})")

    # 결정 사항에 논의 안건이 섞였는지
    for d in gold["discussed"]:
        num = re.search(r"(\d+)번", d["what"])
        if num and re.search(rf"{num.group(1)}번[^\n]*", dec_block):
            issues.append("논의 안건이 결정 사항에 포함")
            break

    # 사람은 원문과 나란히 놓고 이것들도 봅니다. 1차 대조가 본 것과 같은 항목입니다
    if norm(gold["date"]) not in norm(out):
        issues.append("일시가 원문과 다름")
    if {p for p in PEOPLE if p in out} - set(gold["attendees"]):
        issues.append("원문에 없는 인명")
    if set(gold["attendees"]) - {p for p in PEOPLE if p in out}:
        issues.append("참석자 누락")
    for d in gold["decided"]:
        if d["deadline"] != "미정":
            continue
        line = next((l for l in dec_block.split("\n") if d["what"] in l), "")
        if re.search(r"기한:\s*[^\n]*\d", line):
            issues.append("없는 기한을 적음")
            break

    return {"ok": len(issues) == 0, "issue": "; ".join(issues)}


review_results = {i: auto_review(it) for i, it in enumerate(review_queue)}

In [30]:
# 두 더미를 나누어 셉니다. 섞어서 봤지만 수치는 따로 읽어야 합니다
passed_idx = [i for i, it in enumerate(review_queue) if it["review_group"] == "통과"]
flag_idx = [i for i, it in enumerate(review_queue) if it["review_group"] == "표시"]

n_rev = len(passed_idx)
n_bad = sum(1 for i in passed_idx if not review_results[i]["ok"])
rate = n_bad / n_rev
margin = 1.96 * np.sqrt(rate * (1 - rate) / n_rev)

print(f"[통과군 표본] 검수 {n_rev}건 중 불량 {n_bad}건")
print(f"추정 불량률 : {rate:.1%} "
      f"(95% 신뢰구간 {max(0,rate-margin):.1%} ~ {min(1,rate+margin):.1%})")

if flag_idx:
    n_flag = len(flag_idx)
    n_ok = sum(1 for i in flag_idx if review_results[i]["ok"])
    false_alarm = n_ok / n_flag
    print(f"\n[표시군(검토 대상) 표본] 검수 {n_flag}건 중 멀쩡한 것 {n_ok}건")
    print(f"필터 오탐률 : {false_alarm:.1%} (이만큼은 되살릴 수 있습니다)")
else:
    false_alarm = 0.0
    print("\n[표시군(검토 대상) 표본] 표시된 항목이 없습니다")

issue_types = Counter()
for i in passed_idx:
    r = review_results[i]
    if not r["ok"]:
        for part in r["issue"].split("; "):
            issue_types[part.split("(")[0].strip()] += 1
print(f"\n통과군 표본의 불량 유형 : {dict(issue_types.most_common())}")

[통과군 표본] 검수 24건 중 불량 4건
추정 불량률 : 16.7% (95% 신뢰구간 1.8% ~ 31.6%)

[표시군(검토 대상) 표본] 표시된 항목이 없습니다

통과군 표본의 불량 유형 : {'논의사항 개수 불일치': 3, '결정사항 개수 불일치': 2}


두 지표는 따로 읽습니다.

**불량률**은 통과군 표본에서 산출됩니다. 자동 필터를 통과했는데 사람이 불량으로 본 것의 비율입니다. 이것이 **자동 필터가 놓친 것**이고 데이터셋 카드에 적을 수치입니다.

**오탐률**은 표시군(검토 대상) 표본에서 산출됩니다. 자동 필터가 걸러 냈는데 사람이 보니 멀쩡한 것의 비율입니다.

이 값이 높으면 검사가 지나치게 엄격합니다. 대부분 Step 4.2에서 본 "계산한 값이 환각으로 잡히는" 경우입니다.

오탐률이 높다고 검사를 바로 없애면 안 됩니다. **먼저 그 항목들의 사유를 봅니다.** 사유가 한 종류에 몰려 있으면 그 사유만 예외 처리하면 됩니다.

**적은 계층을 더 뽑고 원래 비율로 되돌리기(이론 4.3절).** 쉬움 10건, 보통 20건, 어려움 20건을 따로 뽑아(불비례 배분) 같은 검수(`auto_review`)를 하고 전체 불량률은 원래 비율로 되돌려 계산합니다.

In [31]:
# 이론 4.3절 : 적은 계층을 비율보다 많이 뽑고(불비례 배분) 전체 불량률은 원래 비율로 되돌립니다
over_alloc = {"easy": 10, "normal": 20, "hard": 20}
pop = Counter(it["difficulty"] for it in balanced)        # 원래 비율의 분모
rng = random.Random(45)                                   # 전역 난수 상태를 건드리지 않습니다

per = {}                                                  # 난이도 : [표본 수, 불량 수]
for d, k in over_alloc.items():
    group = [it for it in balanced if it["difficulty"] == d]
    for it in rng.sample(group, min(k, len(group))):
        r = per.setdefault(d, [0, 0])
        r[0] += 1
        r[1] += not auto_review(it)["ok"]

print(f"{'난이도':<8} {'전체':>5} {'표본':>5} {'불량':>5} {'불량률':>7}")
for d in ["easy", "normal", "hard"]:
    n_d, bad_d = per.get(d, [0, 0])
    print(f"{d:<8} {pop[d]:>5} {n_d:>5} {bad_d:>5} {bad_d / max(n_d, 1):>7.1%}")

n_over = sum(n for n, _ in per.values())
naive = sum(b for _, b in per.values()) / n_over
weighted = sum(pop[d] / len(balanced) * b / n for d, (n, b) in per.items())
print(f"\n표본 그대로 계산한 불량률  : {naive:.1%}  (어려움 비중이 원래보다 큼)")
print(f"원래 비율로 되돌린 불량률  : {weighted:.1%}")
print(f"위의 비례 배분 표본 불량률 : {rate:.1%}")

난이도         전체    표본    불량     불량률
easy         9     9     0    0.0%
normal      15    15     3   20.0%
hard         6     6     2   33.3%

표본 그대로 계산한 불량률  : 16.7%  (어려움 비중이 원래보다 큼)
원래 비율로 되돌린 불량률  : 16.7%
위의 비례 배분 표본 불량률 : 16.7%


표본 그대로 계산한 값은 어려움을 원래보다 많이 넣었으므로 어려움의 불량률 쪽으로 치우칩니다. 전체를 말할 때는 원래 비율로 되돌린 값을 쓰고 고칠 곳을 찾을 때는 난이도별 불량률을 봅니다. 위의 비례 배분 표본 불량률(`rate`)은 그대로 전체 추정값이라 카드에 적은 값도 그대로입니다.

어려움 표본이 11건에서 20건이 되면 어려움 불량률의 오차 범위가 약 ±22퍼센트포인트에서 ±9퍼센트포인트로 줄어듭니다(어려움 24건에 대한 유한 모집단 보정 포함).

불량 유형으로는 결정과 논의의 구분 오류가 주로 나올 것입니다. 형식 검사로도 대조 검사로도 안 잡히고 2차 판정도 놓친, **구조적 판단이 필요한 오류**입니다.

In [32]:
# 실습 : 표본 검수에서 발견된 '결정/논의 개수 불일치 및 혼동'을 잡는 새 필터(filter_structure)를 완성하세요

def filter_structure(item):
    """2026년 9월 표본 검수에서 결정과 논의가 섞인 항목이 발견되어 추가한 검사입니다"""
    issues = []
    out, gold = item["output"], item["gold"]

    dec_block = extract_block(out, "## 결정 사항", "## 논의 사항")
    dis_block = extract_block(out, "## 논의 사항", "## 후속 조치")

    n_dec = len([l for l in dec_block.split("\n")
                 if re.match(r"^(\d+\.|-)\s+", l.strip()) and "없음" not in l])
    n_dis = len([l for l in dis_block.split("\n")
                 if re.match(r"^(\d+\.|-)\s+", l.strip()) and len(l.strip()) > 3 and "없음" not in l])

    ### START CODE HERE ### (결정 사항 개수 불일치 및 논의 사항 개수 불일치 조건 2줄)
    if len(gold["decided"]) and n_dec != len(gold["decided"]):
        issues.append("결정사항개수불일치")

    if len(gold["discussed"]) and n_dis != len(gold["discussed"]):
        issues.append("논의사항개수불일치")
    ### END CODE HERE ###

    # 논의 안건 번호가 결정 사항 블록에 섞여 있는지 검사 (미리 작성됨)
    for d in gold["discussed"]:
        num = re.search(r"(\d+)번", d["what"])
        if num and f"{num.group(1)}번" in dec_block:
            issues.append("결정논의혼동")
            break

    return issues


before = len(balanced)
curated = [it for it in balanced if not filter_structure(it)]
# 만약 엄격한 줄 수 비교로 데이터가 너무 적게 남으면 '결정논의혼동'이 없는 항목은 유지합니다
if len(curated) < max(15, int(before * 0.4)):
    curated = [it for it in balanced if "결정논의혼동" not in filter_structure(it)]
print(f"구조 필터 적용 : {before} -> {len(curated)}건 ({len(curated)/max(before,1):.1%} 통과)")

ratio, counts = current_dist(curated)
print(f"분포 : { {k: round(v,3) for k,v in ratio.items()} }, 개수 {counts}")

구조 필터 적용 : 30 -> 25건 (83.3% 통과)
분포 : {'normal': 0.48, 'easy': 0.36, 'hard': 0.16}, 개수 {'normal': 12, 'easy': 9, 'hard': 4}


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
if len(gold["decided"]) and n_dec != len(gold["decided"]):
    issues.append("결정사항개수불일치")

if len(gold["discussed"]) and n_dis != len(gold["discussed"]):
    issues.append("논의사항개수불일치")
```

이론 4.4절의 **검수 결과를 필터에 되돌리는** 작업입니다. 사람이 발견한 오류 유형을 검사로 만들었으니 이제 그 유형은 자동으로 걸러지고 다음 생성에서는 검수 부담이 줄어듭니다. 컨텍스트와 하네스 노트에서 배운 피드백처럼 센서가 놓친 것을 사람이 잡고 그 정보로 센서를 강화하는 루프입니다.

**함수의 docstring에 이 규칙을 왜 추가했는지 명시해 두었습니다.** 이론 4.4절에서 살펴본 것처럼 반년 뒤에는 이 검사가 통과율만 낮추는 불필요한 코드로 오해받아 삭제되기 쉬운데 그 순간 동일한 불량이 데이터셋에 다시 유입되기 때문입니다.

검사를 강화하면 통과 건수가 줄어 목표 개수를 다시 못 채울 수 있습니다. 그러면 Step 5의 재조정을 다시 실행합니다. **파이프라인은 한 방향으로만 흐르지 않습니다.**

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 7. 출처 단위 분할 및 데이터셋 카드 생성

교재 : 4.5.4 지시 파인 튜닝 데이터셋 자동 생성, 4.3.7 데이터 품질

이론 6.5절의 **출처 단위 분할**과 6.4절의 데이터셋 카드를 만듭니다. 카드에는 이론 5.3절이 요구한 **조건 확인 다섯 줄**도 넣습니다. 어떤 모델로 만들었는지, 조건 문서를 언제 확인했는지, 시드가 어디서 왔는지, 어디까지 쓸 계획인지, 법무에 무엇을 물었는지입니다. 출처 단위 분할은 교재 4.3.7이 중복 제거의 이유로 든 훈련-테스트 중첩을 막는 일입니다.

In [33]:
# 실습 : 출처(source_seed) 단위로 학습/검증/테스트 셋을 분할하고 데이터셋 카드를 발행하세요

def split_by_source(items, ratios=(0.7, 0.15, 0.15)):
    """출처 단위로 학습/검증/테스트를 분할하세요"""
    sources = sorted({it["source_seed"] for it in items})
    random.Random(7).shuffle(sources)

    n = len(sources)
    n_tr = max(1, int(n * ratios[0]))
    n_va = max(1, int(n * ratios[1]))
    groups = {"train": set(sources[:n_tr]),
              "valid": set(sources[n_tr:n_tr + n_va]),
              "test": set(sources[n_tr + n_va:])}

    ### START CODE HERE ### (출처 집합 기준 분할 딕셔너리 생성 및 분할 간 출처 교집합 검증)
    # 1) 각 분할(k)의 출처 집합(v)에 it["source_seed"] 가 속하는 항목들로 리스트를 만듭니다
    out = {k: [it for it in items if it["source_seed"] in v] for k, v in groups.items()}

    # 2) 두 분할(a, b) 사이에 겹치는 source_seed 가 없는지 교집합(&)으로 누수를 검증합니다
    for a in out:
        for b in out:
            if a >= b:
                continue
            overlap = ({it["source_seed"] for it in out[a]} &
                       {it["source_seed"] for it in out[b]})
            assert not overlap, f"{a}/{b} 출처 누수 {overlap}"
    ### END CODE HERE ###
    return out


def make_dataset_card(name, version, items, splits, stats):
    """데이터셋 카드를 생성합니다 (미리 작성됨)"""
    ratio, counts = current_dist(items)
    card = {
        "name": name,
        "version": version,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "task": TASK,
        "generation": {
            "model": GEN_MODEL,
            "revision": GEN_REVISION,
            "temperature": 0.7,
            "prompt_version": "v1",
            "instruction_variants": len(INSTRUCTION_VARIANTS),
            "generated_total": stats["generated"],
            "libraries": {p: md.version(p) for p in ["transformers", "peft", "trl"]},
        },
        "filters": {
            "1차_형식": "4개 섹션, 담당/기한 표기, 개요 필드",
            "1차_대조": "일시 일치, 인명 환각, 숫자 환각, 참석자 누락",
            "1차_중복": "MinHash 임계값 0.92 (숫자/인명 치환 후)",
            "2차_판정": f"{JUDGE_MODEL}, 예/아니오 {len(JUDGE_CHECKS)}항목, 판정불가는 탈락",
            "3차_환류": "결정/논의 개수 일치, 혼동 검사 (표본 검수에서 발견해 추가)",
        },
        "license_check": {
            "생성_및_판정_모델": f"{GEN_MODEL} ({GEN_REVISION}), 판정: {JUDGE_MODEL}",
            "조건_문서와_확인일": ("https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/blob/main/LICENSE"
                           " (Apache-2.0), 2026-09-18 확인"),
            "시드_출처": "규칙으로 생성한 가상 회의록 (실제 사내 문서 미사용)",
            "사용_범위": "사내 전용. 자회사 배포나 데이터 공개는 재확인 필요",
            "법무_문의": "미문의. 배포 계획이 생기면 이론 5.2절 세 질문으로 문의",
        },
        "stats": {
            "final_count": len(items),
            "pass_rate": round(len(items) / max(stats["generated"], 1), 3),
            "stage_excluded": stats["stage_excluded"],
            "difficulty_distribution": {k: round(v, 3) for k, v in ratio.items()},
            "difficulty_counts": counts,
        },
        "splits": {k: {"count": len(v),
                       "sources": len({it['source_seed'] for it in v})}
                   for k, v in splits.items()},
        "split_policy": "출처(source_seed) 단위 분할. 항목 단위 누수 없음",
        "review": {
            "sample_size": n_rev,
            "defect_rate": round(rate, 3),
            "confidence_interval_95": [round(max(0, rate - margin), 3),
                                       round(min(1, rate + margin), 3)],
            "defect_types": dict(issue_types.most_common()),
            "flagged_reviewed": len(flag_sample),
            "flagged_false_alarm_rate": round(false_alarm, 3),
        },
        "known_limitations": [
            "어려움 난이도는 통과율이 가장 낮아 재생성으로 채웁니다. 실제 어려운 사례의 다양성이 부족할 수 있습니다.",
            "생성 모델이 3B급이므로 복잡한 조건부 기한 처리에서 오류 가능성이 있습니다.",
            "회의록 원문이 규칙으로 만든 것이므로 실제 회의록의 문체 다양성을 반영하지 않습니다.",
            f"검수 표본 {n_rev}건 기준 불량률 {rate:.1%}. 전수 검수하지 않았습니다.",
            f"2차 판정의 자기 출력 편향을 {SELF_N}건으로만 확인했습니다. "
            f"관대함 차이 {r_self['관대함'] - r_other['관대함']:+.0%}.",
        ],
    }
    return card


splits = split_by_source(curated)
print("분할 결과")
for k, v in splits.items():
    print(f"  {k:<7} {len(v):>4}건, 출처 {len({it['source_seed'] for it in v}):>4}개, "
          f"난이도 {dict(Counter(it['difficulty'] for it in v))}")

stats = {"generated": len(raw_dataset) + extra_generated,
         "stage_excluded": {k: len(v) for k, v in stages.items() if k != "passed"}}
card = make_dataset_card("meeting_minutes_format", "v1.0.0", curated, splits, stats)

Path("dataset_card.json").write_text(json.dumps(card, ensure_ascii=False, indent=2),
                                     encoding="utf-8")
for k, v in splits.items():
    Path(f"ds_{k}.json").write_text(json.dumps(v, ensure_ascii=False, indent=2),
                                    encoding="utf-8")
print("\n데이터셋 카드와 분할 파일 저장 완료")

분할 결과
  train     17건, 출처   17개, 난이도 {'easy': 7, 'normal': 8, 'hard': 2}
  valid      3건, 출처    3개, 난이도 {'easy': 1, 'hard': 1, 'normal': 1}
  test       5건, 출처    5개, 난이도 {'normal': 3, 'easy': 1, 'hard': 1}

데이터셋 카드와 분할 파일 저장 완료


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
out = {k: [it for it in items if it["source_seed"] in v] for k, v in groups.items()}
# ...
overlap = ({it["source_seed"] for it in out[a]} &
           {it["source_seed"] for it in out[b]})
```

분할 후에는 `assert`로 누수를 검증했습니다. 코드가 맞다고 믿지 말고 **분할 후 반드시 검사합니다.**

`license_check` 다섯 줄은 이론 5.3절이 요구한 것입니다. 지금은 사내 전용이라 걸리는 것이 없지만 **자회사에 넘기는 순간 배포가 되고 조건이 붙습니다.** 그때 이 다섯 줄이 있으면 확인이 몇 분에 끝나고 없으면 처음부터 다시 조사해야 합니다.

**[현업 환경 적용]** : `시드_출처`와 `법무_문의`는 사내에서 값이 달라지는 항목입니다. 사내 회의록을 시드로 썼다면 그 폴더 경로와 반출 등급을 적고 법무에 물었다면 문의 날짜와 답을 받은 날짜를 적습니다.

이 카드에서는 `known_limitations`가 가장 값집니다. 나중에 모델이 복잡한 조건부 기한에서 실패하면 카드를 보고 데이터셋의 알려진 한계임을 즉시 알 수 있어 원인 추적에 며칠을 쓰지 않습니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

In [34]:
print(json.dumps(card["stats"], ensure_ascii=False, indent=2))
print(json.dumps(card["license_check"], ensure_ascii=False, indent=2))
print(json.dumps(card["known_limitations"], ensure_ascii=False, indent=2))

{
  "final_count": 25,
  "pass_rate": 0.532,
  "stage_excluded": {
    "format": 13,
    "fact": 0,
    "dup": 0,
    "judge": 0
  },
  "difficulty_distribution": {
    "normal": 0.48,
    "easy": 0.36,
    "hard": 0.16
  },
  "difficulty_counts": {
    "normal": 12,
    "easy": 9,
    "hard": 4
  }
}
{
  "생성_및_판정_모델": "Qwen/Qwen2.5-3B-Instruct ((확인 못 함)), 판정: unsloth/Llama-3.2-3B-Instruct",
  "조건_문서와_확인일": "https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/blob/main/LICENSE (Apache-2.0), 2026-09-18 확인",
  "시드_출처": "규칙으로 생성한 가상 회의록 (실제 사내 문서 미사용)",
  "사용_범위": "사내 전용. 자회사 배포나 데이터 공개는 재확인 필요",
  "법무_문의": "미문의. 배포 계획이 생기면 이론 5.2절 세 질문으로 문의"
}
[
  "어려움 난이도는 통과율이 가장 낮아 재생성으로 채웁니다. 실제 어려운 사례의 다양성이 부족할 수 있습니다.",
  "생성 모델이 3B급이므로 복잡한 조건부 기한 처리에서 오류 가능성이 있습니다.",
  "회의록 원문이 규칙으로 만든 것이므로 실제 회의록의 문체 다양성을 반영하지 않습니다.",
  "검수 표본 24건 기준 불량률 16.7%. 전수 검수하지 않았습니다.",
  "2차 판정의 자기 출력 편향을 8건으로만 확인했습니다. 관대함 차이 +0%."
]


**교재 4.5.4 형식으로 내보내기와 시점 기록.** 학습 도구에 넘길 파일에는 `instruction`, `input`, `output` 세 필드만 둡니다. 교재 4.5.4가 예로 든 JSON 형식입니다. `gold`, `status` 같은 추적용 필드는 `ds_*.json`에 그대로 남깁니다. 같은 절의 "동적으로 변하는 데이터" 권고대로 카드에는 원문이 어느 시점의 것인지도 적습니다(이론 6.4절). 만든 시각은 이미 카드의 `created_at`에 있습니다.

In [35]:
# 교재 4.5.4 : 학습용 파일은 {"instruction", "input", "output"} 세 필드만 한 줄에 하나씩 씁니다
for k, v in splits.items():
    rows = [{f: it[f] for f in ("instruction", "input", "output")} for it in v]
    Path(f"sft_{k}.jsonl").write_text(
        "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")

# 교재 4.5.4 동적 데이터 : 원문이 어느 시점의 것인지 카드에 남깁니다(만든 시각은 created_at)
dates = sorted((int(m), int(d)) for m, d in
               (re.findall(r"\d+", it["gold"]["date"]) for it in curated))
card["data_time"] = {
    "source_snapshot": "make_minutes 규칙 원문 (Step 3 seed 1000~1239, Step 5 seed 90000~)",
    "source_window": f"{dates[0][0]}월 {dates[0][1]}일 ~ {dates[-1][0]}월 {dates[-1][1]}일 (연도 없음)",
    "refresh_rule": "원문이 바뀌면 새 버전으로 다시 만들고 이전 버전은 롤백용으로 보관",
}
Path("dataset_card.json").write_text(json.dumps(card, ensure_ascii=False, indent=2),
                                     encoding="utf-8")

print(json.dumps(card["data_time"], ensure_ascii=False, indent=2))
for k in splits:
    n_lines = len(Path(f"sft_{k}.jsonl").read_text(encoding="utf-8").splitlines())
    print(f"sft_{k}.jsonl : {n_lines}줄")
print(Path("sft_train.jsonl").read_text(encoding="utf-8").splitlines()[0][:150])

{
  "source_snapshot": "make_minutes 규칙 원문 (Step 3 seed 1000~1239, Step 5 seed 90000~)",
  "source_window": "1월 13일 ~ 11월 23일 (연도 없음)",
  "refresh_rule": "원문이 바뀌면 새 버전으로 다시 만들고 이전 버전은 롤백용으로 보관"
}
sft_train.jsonl : 17줄
sft_valid.jsonl : 3줄
sft_test.jsonl : 5줄
{"instruction": "회의록을 표준 서식으로 재작성하세요.", "input": "7월 14일 데이터 이관 회의. 참석 최유나, 정하윤, 김서준.\n- 1. 데이터 이관 1번 안건 추진 확정. 정하윤 담당, 기한 8월 14일.\n- 2. 데이터 이관 2번 안건 


**[현업 환경 적용]** : 사내에서는 `source_snapshot`에 파이프라인 노트에서 만든 스냅샷의 이름과 날짜를, `source_window`에 원문 문서의 수정일 범위를 적습니다.

## Step 8. SFT 학습 A/B 비교를 통한 큐레이션 효과 증명

교재 : 5.5.6 지시 튜닝과 인간 피드백 기반 강화 학습

이론 6.6절에서 통과율 같은 수치는 대리 지표이고 최종 증명은 학습 결과라고 했습니다. 여기서 하는 학습이 교재 5.5.6의 지시 튜닝(SFT)입니다. A/B 비교를 합니다.

고정할 것은 베이스 모델, 학습 설정, **데이터 개수**, 테스트 세트 넷입니다. 이 가운데 데이터 개수가 가장 자주 빠집니다. 그래서 큐레이션한 데이터셋과 **같은 크기로** 원본에서 무작위 표집합니다.

In [36]:
# 두 데이터셋 준비
curated_train = splits["train"]
n_train = len(curated_train)

# 원본(필터 미적용)에서 같은 개수를 뽑습니다.
# 테스트와 검증에 쓰인 출처는 빼야 합니다. 안 빼면 원본 쪽만 테스트 문제를 미리 본 셈이 됩니다
held_out = {it["source_seed"] for k in ("valid", "test") for it in splits[k]}
raw_pool = [it for it in raw_dataset if it["source_seed"] not in held_out]
uncurated_train = random.Random(11).sample(raw_pool, min(n_train, len(raw_pool)))

print(f"큐레이션 학습셋 : {len(curated_train)}건")
print(f"원본 학습셋     : {len(uncurated_train)}건 (후보 {len(raw_pool)}건에서)")
print(f"원본셋의 필터 통과 비율 : "
      f"{sum(1 for it in uncurated_train if it.get('status')=='passed')/len(uncurated_train):.1%}")

큐레이션 학습셋 : 17건
원본 학습셋     : 17건 (후보 36건에서)
원본셋의 필터 통과 비율 : 58.8%


**개수를 맞췄습니다.** 이론 6.6절에서 가장 자주 빠진다고 한 항목입니다. 큐레이션 쪽이 78건인데 원본 쪽을 240건 전부 쓰면 차이가 품질 때문인지 개수 때문인지 알 수 없습니다.

**테스트와 검증에 쓰인 출처를 뺐습니다.** 원본 쪽을 `raw_dataset` 전체에서 뽑으면 테스트 세트와 같은 출처가 섞여 들어와 원본 쪽만 테스트 문제를 미리 본 상태로 비교하게 됩니다. 이론 6.5절의 누수가 A/B 비교에서는 이렇게 나타납니다.

**원본셋의 필터 통과 비율**이 100퍼센트가 아닌지도 반드시 확인합니다. 100퍼센트면 원본이 아니라 큐레이션 데이터를 두 번 쓴 것이고 그 비교는 아무것도 증명하지 못합니다.

In [37]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

BASE = "Qwen/Qwen2.5-1.5B-Instruct"       # A/B 두 번 학습하므로 작은 모델로


def to_chat(items):
    # 학습과 추론의 프롬프트 형식을 맞춥니다
    return [{"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{it['instruction']}\n\n{it['input']}"},
        {"role": "assistant", "content": it["output"]},
    ]} for it in items]


def train_on(items, tag, epochs=1):
    tok = AutoTokenizer.from_pretrained(BASE)
    tok.pad_token = tok.pad_token or tok.eos_token

    # T4 GPU에서 BFloat16 GradScaler 충돌(_amp_foreach_non_finite_check_and_unscale_cuda)을
    # 원천 차단하기 위해 학습용 4-bit 설정의 연산 타입을 float32로 두고 fp16/bf16 스케일러를 끕니다
    bnb_train = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float32,
        bnb_4bit_use_double_quant=True,
    )
    m = AutoModelForCausalLM.from_pretrained(
        BASE, quantization_config=bnb_train, dtype=torch.float32, device_map="auto"
    )
    m.config.torch_dtype = torch.float32
    if getattr(m, "generation_config", None) is not None:
        m.generation_config.max_length = None

    m = prepare_model_for_kbit_training(m)
    m = get_peft_model(m, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none", task_type="CAUSAL_LM"))

    # LoRA 어댑터 및 비양자화 레이어에 남아 있을 수 있는 BFloat16/Float16 텐서를 모두 Float32로 통일합니다
    for p in m.parameters():
        if p.requires_grad or p.dtype in (torch.bfloat16, torch.float16):
            p.data = p.data.to(torch.float32)
    for b in m.buffers():
        if b.dtype in (torch.bfloat16, torch.float16):
            b.data = b.data.to(torch.float32)

    args = SFTConfig(output_dir=f"./ab_{tag}", num_train_epochs=epochs,
                     per_device_train_batch_size=1, gradient_accumulation_steps=4,
                     learning_rate=2e-4, logging_steps=10,
                     fp16=False, bf16=False,
                     gradient_checkpointing=True, max_length=768,
                     report_to="none", save_strategy="no")
    tr = SFTTrainer(model=m, args=args,
                    train_dataset=Dataset.from_list(to_chat(items)),
                    processing_class=tok)
    for p in tr.model.parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.to(torch.float32)
    tr.train()
    m.eval()
    return tok, m

In [38]:
# 평가 함수. 난이도별로 나누어 측정합니다
def evaluate_model(tok, model, test_items):
    per_diff = {}
    if getattr(model, "generation_config", None) is not None:
        model.generation_config.max_length = None
    for it in test_items:
        prompt = f"{it['instruction']}\n\n{it['input']}"
        msgs = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}]
        ids = tok(tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False),
                  return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=320, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        ans = tok.decode(out[0][ids["input_ids"].shape[1]:],
                         skip_special_tokens=True,
                         clean_up_tokenization_spaces=False)

        cand = {**it, "output": ans}
        fmt_ok = not filter_format(cand)
        fact_ok = not filter_fact(cand)
        struct_ok = not filter_structure(cand)

        d = per_diff.setdefault(it["difficulty"],
                               {"n": 0, "format": 0, "fact": 0, "structure": 0, "all": 0})
        d["n"] += 1
        d["format"] += fmt_ok
        d["fact"] += fact_ok
        d["structure"] += struct_ok
        d["all"] += (fmt_ok and fact_ok and struct_ok)
    return per_diff


def print_eval(name, per_diff):
    print(f"[{name}]")
    print(f"  {'난이도':<10} {'n':>4} {'형식':>7} {'사실':>7} {'구조':>7} {'전체':>7}")
    tot = {"n": 0, "format": 0, "fact": 0, "structure": 0, "all": 0}
    for d in ["easy", "normal", "hard"]:
        if d not in per_diff:
            continue
        v = per_diff[d]
        for k in tot:
            tot[k] += v[k]
        print(f"  {d:<10} {v['n']:>4} {v['format']/v['n']:>6.0%} {v['fact']/v['n']:>6.0%} "
              f"{v['structure']/v['n']:>6.0%} {v['all']/v['n']:>6.0%}")
    print(f"  {'합계':<10} {tot['n']:>4} {tot['format']/max(tot['n'],1):>6.0%} "
          f"{tot['fact']/max(tot['n'],1):>6.0%} {tot['structure']/max(tot['n'],1):>6.0%} "
          f"{tot['all']/max(tot['n'],1):>6.0%}\n")
    return tot["all"] / max(tot["n"], 1)

In [39]:
# A : 원본 데이터셋으로 학습
tok_a, model_a = train_on(uncurated_train, "uncurated")
eval_a = evaluate_model(tok_a, model_a, splits["test"])
score_a = print_eval("A. 원본 데이터셋", eval_a)

del model_a
gc.collect(); torch.cuda.empty_cache()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

{'train_runtime': '33.01', 'train_samples_per_second': '0.515', 'train_steps_per_second': '0.151', 'train_loss': '1.182', 'entropy': '1.183', 'mean_token_accuracy': '0.7583', 'num_tokens': '7270', 'epoch': '1'}
[A. 원본 데이터셋]
  난이도           n      형식      사실      구조      전체
  easy          1     0%     0%     0%     0%
  normal        3     0%    33%     0%     0%
  hard          1     0%     0%     0%     0%
  합계            5     0%    20%     0%     0%



In [40]:
# B : 큐레이션 데이터셋으로 학습
tok_b, model_b = train_on(curated_train, "curated")
eval_b = evaluate_model(tok_b, model_b, splits["test"])
score_b = print_eval("B. 큐레이션 데이터셋", eval_b)

del model_b
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

{'train_runtime': '33.81', 'train_samples_per_second': '0.503', 'train_steps_per_second': '0.148', 'train_loss': '1.238', 'entropy': '1.224', 'mean_token_accuracy': '0.7447', 'num_tokens': '6944', 'epoch': '1'}
[B. 큐레이션 데이터셋]
  난이도           n      형식      사실      구조      전체
  easy          1     0%     0%     0%     0%
  normal        3     0%    33%     0%     0%
  hard          1     0%     0%     0%     0%
  합계            5     0%    20%     0%     0%



In [41]:
print(f"{'조건':<24} {'전체 통과율':>12}")
print("-" * 40)
print(f"{'A. 원본 (필터 미적용)':<24} {score_a:>11.0%}")
print(f"{'B. 큐레이션':<24} {score_b:>11.0%}")
print(f"{'차이':<24} {score_b - score_a:>+11.0%}")

조건                             전체 통과율
----------------------------------------
A. 원본 (필터 미적용)                    0%
B. 큐레이션                           0%
차이                               +0%


**같은 개수, 같은 하이퍼파라미터, 같은 평가 세트입니다.** 달라진 것은 데이터 품질뿐입니다.

차이가 나왔다면 대리 지표(통과율, 불량률)가 좋았던 데이터셋이 실제 학습 결과도 좋았다는 뜻입니다. 이론 6.6절이 말한 최종 증명을 얻었습니다.

차이가 작거나 없는 것도 의미 있는 결과입니다. 데이터가 120건으로 너무 적어 두 조건 모두 충분히 학습되지 않았거나 원본의 불량 비율이 낮아 차이가 만들어지지 않았거나 평가 세트가 작아 차이가 드러나지 않았을 수 있습니다. **어느 쪽인지 판단해 리포트에 적습니다.**

난이도별 표도 반드시 봅니다. 전체 평균은 비슷한데 어려움에서만 차이가 나는 경우가 있어 이론 6.6절에서 난이도별로 나눠 보라고 했습니다.

## Step 9. 선택 심화 트랙

여섯 트랙 중 **두 개를 선택**해 수행하고 공유합니다.

<div style="font-size: 18px; line-height: 1.6;">

| 트랙 | 주제 | 성격 | 난이도 |
|---|---|---|---|
| 1 | 데이터 규모와 성능의 관계 | 측정 | 중 |
| 2 | 생성 프롬프트 버전 비교 | 개선 | 중 |
| 3 | 어려운 사례 집중 학습 | 실험 | 중 |
| 4 | 다양성 지표 측정과 개선 | 측정 | 상 |
| 5 | 데이터셋 버전 관리 자동화 | 확장 | 중 |
| 6 | 문서 단위 질문 생성과 청크별 생성 비교 | 비교 | 중 |

</div>

#### 트랙 1. 학습 데이터 규모별 SFT 성능 변화 측정

QLoRA 실습의 트랙 2와 같은 실험을 이번에는 **큐레이션 여부와 함께** 봅니다.

In [43]:
# 실습 : 데이터 규모(15건, 전체)를 바꿔가며 큐레이션 유무별 학습 성능을 비교하세요

### START CODE HERE ### (실험할 규모 목록 sizes 정의 및 표본 추출)
sizes = sorted({min(15, len(curated_train)), len(curated_train)})
### END CODE HERE ###

results = {}
for label, pool in [("curated", curated_train), ("uncurated", uncurated_train)]:
    for n in sizes:
        if n > len(pool) or n <= 0:
            continue
        ### START CODE HERE ### (pool에서 시드 3으로 n건을 무작위 비복원 추출합니다)
        subset = random.Random(3).sample(pool, n)
        ### END CODE HERE ###
        tok_x, model_x = train_on(subset, f"{label}_{n}", epochs=1)
        ev = evaluate_model(tok_x, model_x, splits["test"])
        tot = sum(v["all"] for v in ev.values()) / max(sum(v["n"] for v in ev.values()), 1)
        results[(label, n)] = tot
        print(f"{label:<12} n={n:<4} 전체 {tot:.0%}")
        del model_x
        gc.collect(); torch.cuda.empty_cache()

print(f"\n{'규모':>6} {'원본':>10} {'큐레이션':>10} {'차이':>8}")
print("-" * 38)
for n in sizes:
    u = results.get(("uncurated", n))
    c = results.get(("curated", n))
    if u is not None and c is not None:
        print(f"{n:>6} {u:>9.0%} {c:>9.0%} {c-u:>+8.0%}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

...

Tokenizing train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

{'train_runtime': '27.56', 'train_samples_per_second': '0.617', 'train_steps_per_second': '0.181', 'train_loss': '1.178', 'entropy': '1.185', 'mean_token_accuracy': '0.7535', 'num_tokens': '7270', 'epoch': '1'}
uncurated    n=17   전체 0%

    규모         원본       큐레이션       차이
--------------------------------------
    15        0%        0%      +0%
    17        0%        0%      +0%


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
sizes = sorted({40, 80, len(curated_train)})
# ...
subset = random.Random(3).sample(pool, n)
```

**차이가 규모에 따라 어떻게 변하는지** 봅니다.

데이터가 적을 때는 불량 하나가 큰 영향을 주지만 많아지면 다수의 정상 데이터가 상쇄합니다. 대개 소규모에서 차이가 크고 규모가 커지면 줄어듭니다.

그래서 **데이터가 적을 때 큐레이션이 특히 중요합니다.** QLoRA 노트에서 다룬 적은 데이터를 잘 만들어 먼저 학습시키는 전략에서는 큐레이션이 필수입니다. 수만 건을 쓸 계획이면 큐레이션에 드는 비용과 효과를 비교해야 합니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 2. 생성 프롬프트 개선 전후의 통과율 및 학습 성능 비교

생성 프롬프트를 개선해 필터 통과율을 높입니다. 데이터 엔지니어링 노트의 트랙과 비슷하지만 여기서는 **학습 결과까지** 확인합니다.

In [42]:
# 실습 : 실패 유형을 근거로 생성 프롬프트(GEN_PROMPT_V2)를 개선하고 통과율과 학습 성능을 비교하세요

print("현재 실패 유형 :", dict(reasons.most_common()))

ADDED_V2_RULES = """5. 결정 사항과 논의 사항의 개수를 원문의 안건 수와 정확히 일치시키세요.
   원문에서 '확정'이라고 명시된 것만 결정 사항입니다.
   '검토', '재논의', '의견 수렴', '이월'이 언급된 안건은 모두 논의 사항입니다.
6. 참석자는 원문에 적힌 인원 전체를 빠짐없이 나열하세요.
7. 원문에 없는 인명, 날짜, 금액을 절대 만들지 마세요.
8. 정리본만 출력하고 설명이나 코드블록 표시를 붙이지 마세요."""

### START CODE HERE ### (GEN_PROMPT 의 기존 5번 문장을 ADDED_V2_RULES 로 교체합니다)
GEN_PROMPT_V2 = GEN_PROMPT.replace(
    "5. 정리본만 출력하고 설명을 붙이지 마세요.",
    ADDED_V2_RULES
)
### END CODE HERE ###


def generate_batch(prompt_tpl, n, start_seed, dist=TARGET_DIST):
    plan_local = []
    for d, r in dist.items():
        plan_local += [d] * max(1, int(n * r))
    random.shuffle(plan_local)

    out = []
    for i, d in enumerate(plan_local):
        m = make_minutes(start_seed + i, d)
        gen = generate(prompt_tpl.format(spec=FORMAT_SPEC, raw=m["raw"]))
        out.append({"id": start_seed + i, "instruction": random.choice(INSTRUCTION_VARIANTS),
                    "input": m["raw"], "output": gen.strip(), "gold": m["gold"],
                    "difficulty": d, "source_seed": start_seed + i})
    return out


v2 = generate_batch(GEN_PROMPT_V2, 30, 300000)
st2, rs2 = run_filters(v2)
v2_curated = [it for it in st2["passed"] if "결정논의혼동" not in filter_structure(it)]

print(f"\nv1 통과율 : {len(curated)/max(len(raw_dataset),1):.1%}")
print(f"v2 통과율 : {len(v2_curated)/max(len(v2),1):.1%}")
print(f"v2 실패 유형 : {dict(rs2.most_common())}")

if len(v2_curated) >= 10:
    tok_v2, model_v2 = train_on(v2_curated[:len(curated_train)], "v2prompt", epochs=1)
    ev = evaluate_model(tok_v2, model_v2, splits["test"])
    s = print_eval("v2 프롬프트 데이터셋", ev)
    print(f"v1 큐레이션 : {score_b:.0%}  vs  v2 프롬프트 : {s:.0%}")
    del model_v2
    gc.collect(); torch.cuda.empty_cache()

현재 실패 유형 : {'담당기한누락': 13}

v1 통과율 : 56.8%
v2 통과율 : 60.0%
v2 실패 유형 : {'담당기한누락': 9, '섹션누락': 2}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

{'train_runtime': '27.82', 'train_samples_per_second': '0.611', 'train_steps_per_second': '0.18', 'train_loss': '1.268', 'entropy': '1.25', 'mean_token_accuracy': '0.7406', 'num_tokens': '6591', 'epoch': '1'}
[v2 프롬프트 데이터셋]
  난이도           n      형식      사실      구조      전체
  easy          1     0%     0%     0%     0%
  normal        3     0%    33%     0%     0%
  hard          1     0%     0%     0%     0%
  합계            5     0%    20%     0%     0%

v1 큐레이션 : 0%  vs  v2 프롬프트 : 0%


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
GEN_PROMPT_V2 = GEN_PROMPT.replace(
    "5. 정리본만 출력하고 설명을 붙이지 마세요.",
    ADDED_V2_RULES
)
```

통과율이 올라도 학습 결과는 나빠질 수 있습니다.

첫째, 프롬프트를 강하게 제약하면 출력이 단조로워집니다. 통과율은 오르지만 다양성이 떨어져 모델이 좁게 배웁니다. 트랙 4의 다양성 지표로 확인할 수 있습니다.

둘째, 필터가 잡는 것만 좋아지고 필터가 검사하지 않는 품질 요소는 나빠졌을 수 있습니다. 통과율은 필터 기준의 대리 지표일 뿐입니다.

**그래서 프롬프트 개선의 효과도 이론 6.6절에서 본 대로 학습 결과로 검증합니다.**

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 3. 난이도 구성 비율에 따른 고난도 테스트 성능 비교

난이도 분포를 바꿔 학습하면 어떻게 되는지 확인합니다.

In [ ]:
# 실습 : 세 가지 난이도 분포(쉬움 위주 / 균형 / 어려움 위주)로 학습해 난이도별 성능을 비교하세요

def build_by_dist(pool, dist, n):
    by_diff = {}
    for it in pool:
        by_diff.setdefault(it["difficulty"], []).append(it)
    out = []
    for d, r in dist.items():
        want = int(n * r)
        avail = by_diff.get(d, [])
        random.Random(5).shuffle(avail)
        out += avail[:want]
    random.shuffle(out)
    return out


### START CODE HERE ### (A: 쉬움 0.7/보통 0.3/어려움 0.0, B: TARGET_DIST, C: 쉬움 0.1/보통 0.3/어려움 0.6)
dists = {
    "A_easy_heavy": {"easy": 0.7, "normal": 0.3, "hard": 0.0},
    "B_balanced": TARGET_DIST,
    "C_hard_heavy": {"easy": 0.1, "normal": 0.3, "hard": 0.6},
}
### END CODE HERE ###

table = {}
for tag, dist in dists.items():
    subset = build_by_dist(curated_train, dist, len(curated_train))
    if len(subset) < 8:
        print(f"{tag} : 데이터 부족 ({len(subset)}건). 건너뜁니다")
        continue
    print(f"\n{tag} : {len(subset)}건, {dict(Counter(it['difficulty'] for it in subset))}")
    tok_x, model_x = train_on(subset, tag, epochs=1)
    ev = evaluate_model(tok_x, model_x, splits["test"])
    table[tag] = ev
    print_eval(tag, ev)
    del model_x
    gc.collect(); torch.cuda.empty_cache()

print(f"{'조건':<16} {'easy':>8} {'normal':>8} {'hard':>8}")
print("-" * 44)
for tag, ev in table.items():
    row = []
    for d in ["easy", "normal", "hard"]:
        v = ev.get(d)
        row.append(f"{v['all']/v['n']:.0%}" if v else "-")
    print(f"{tag:<16} " + " ".join(f"{x:>8}" for x in row))


A_easy_heavy : 10건, {'normal': 4, 'easy': 6}

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

...

Tokenizing train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

{'train_runtime': '25.85', 'train_samples_per_second': '0.542', 'train_steps_per_second': '0.155', 'train_loss': '1.326', 'entropy': '1.253', 'mean_token_accuracy': '0.7274', 'num_tokens': '5847', 'epoch': '1'}
[B_balanced]
  난이도           n      형식      사실      구조      전체
  easy          2     0%    50%     0%     0%
  normal        1     0%   100%     0%     0%
  hard          2     0%    50%     0%     0%
  합계            5     0%    60%     0%     0%

C_hard_heavy : 데이터 부족 (7건). 건너뜁니다
조건                   easy   normal     hard
--------------------------------------------
A_easy_heavy           0%       0%       0%
B_balanced             0%       0%       0%

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
dists = {
    "A_easy_heavy": {"easy": 0.7, "normal": 0.3, "hard": 0.0},
    "B_balanced": TARGET_DIST,
    "C_hard_heavy": {"easy": 0.1, "normal": 0.3, "hard": 0.6},
}
```

대개 **학습 분포가 성능 분포를 정합니다.** 쉬움 위주로 학습하면 어려운 입력에서 크게 떨어지고 어려움 위주로 학습하면 어려운 입력에서 나아집니다.

또 어려운 사례를 처리할 수 있으면 쉬운 사례는 자연히 처리되므로 **어려움 위주로 학습해도 쉬운 입력에서 성능이 크게 떨어지지 않는 경우가 많습니다.**

세 조건이 똑같이 나와도 결과입니다. 테스트 세트의 어려움이 네댓 건뿐이라 **차이를 가를 해상도가 없는 것**입니다. 어려움 테스트 항목을 더 만들어 테스트 세트를 키우거나 조건마다 시드를 바꿔 두세 번 실행해 평균을 냅니다.

**표가 평평하다고 "차이가 없다"고 적으면 안 됩니다.** "이 표본 크기로는 구분되지 않았다"가 맞는 문장입니다.

그렇다고 어려움 위주 학습이 항상 낫지는 않습니다. 어려운 데이터는 만들기 비싸고 불량이 섞일 확률도 높습니다(Step 4.5의 난이도별 통과율에서 확인했습니다). **실제 업무 입력 분포에 맞추는 것**이 기본 원칙이고 어려운 쪽으로 약간 치우치게 하는 것이 실용적 조정입니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 4. 데이터셋 다양성 지표(TTR·유사도·4-gram) 측정과 온도별 비교
교재 : 3.5 LLM 기반 애플리케이션에서 무엇을 제어할 수 있는가?, 4.3.7 데이터 품질

합성한 것들이 서로 닮는 문제(이론 2.6절)를 측정하는 지표를 만들고 개선합니다.

In [ ]:
# 실습 : 데이터셋의 다양성 지표(TTR, 평균 임베딩 유사도, 4-gram 중복률, 길이 표준편차)를 구하고 온도별 변화를 측정하세요

from langchain_huggingface import HuggingFaceEmbeddings

div_emb = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-small",
                                model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
                                encode_kwargs={"normalize_embeddings": True})


def diversity_metrics(items, sample=25):
    outs = [it["output"] for it in items[:sample]]
    if len(outs) < 2:
        return {"TTR": 0.0, "평균유사도": 0.0, "4gram중복률": 0.0, "길이표준편차": 0.0}

    ### START CODE HERE ### (1. 어휘 다양성 TTR 계산 및 3. 4-gram 중복률 계산 2줄)
    # 1) 어휘 다양성 (TTR) : 고유 단어 수(len(set(tokens))) / 전체 단어 수(max(len(tokens), 1))
    tokens = [w for o in outs for w in re.findall(r"[가-힣A-Za-z]+", o)]
    ttr = len(set(tokens)) / max(len(tokens), 1)

    # 2) 임베딩 평균 유사도 (구조 제거 후, 미리 작성됨)
    stripped = []
    for o in outs:
        s = re.sub(r"\d+", "N", o)
        for p in PEOPLE:
            s = s.replace(p, "P")
        stripped.append(f"passage: {s}")
    vecs = np.array(div_emb.embed_documents(stripped))
    sim = vecs @ vecs.T
    n = len(vecs)
    mean_sim = (sim.sum() - n) / max(n * (n - 1), 1)

    # 3) 4-gram 중복률 : 1 - (고유 4-gram 수 len(uniq_ng) / 전체 4-gram 수 max(all_ng, 1))
    def ngrams(text, k=4):
        w = re.findall(r"[가-힣A-Za-z]+", text)
        return {tuple(w[i:i+k]) for i in range(len(w) - k + 1)}
    all_ng, uniq_ng = 0, set()
    for o in outs:
        g = ngrams(o)
        all_ng += len(g)
        uniq_ng |= g
    dup_rate = 1 - len(uniq_ng) / max(all_ng, 1)
    ### END CODE HERE ###

    # 4) 길이 분포
    lens = [len(o) for o in outs]

    return {"TTR": round(ttr, 4),
            "평균유사도": round(float(mean_sim), 4),
            "4gram중복률": round(dup_rate, 4),
            "길이표준편차": round(float(np.std(lens)), 1)}


print("v1 큐레이션 데이터셋")
print(diversity_metrics(curated))

for temp in [0.3, 0.7, 1.0]:
    batch = []
    for i in range(8):
        m = make_minutes(500000 + i, "normal")
        o = generate(GEN_PROMPT.format(spec=FORMAT_SPEC, raw=m["raw"]), temperature=temp)
        batch.append({"output": o.strip(), "gold": m["gold"], "input": m["raw"],
                      "difficulty": "normal", "id": i, "source_seed": 500000 + i,
                      "instruction": INSTRUCTION_VARIANTS[0]})
    st, _ = run_filters(batch)
    print(f"\n온도 {temp} : 통과율 {len(st['passed'])/len(batch):.0%}")
    print(f"  {diversity_metrics(batch)}")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB

tokenizer.json: downloading bytes:           |  0.00B

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

v1 큐레이션 데이터셋
{'TTR': 0.077, '평균유사도': 0.9559, '4gram중복률': 0.4458, '길이표준편차': 46.6}

온도 0.3 : 통과율 38%
  {'TTR': 0.1248, '평균유사도': 0.961, '4gram중복률': 0.2765, '길이표준편차': 36.5}

온도 0.7 : 통과율 50%
  {'TTR': 0.1353, '평균유사도': 0.9598, '4gram중복률': 0.2432, '길이표준편차': 32.9}

온도 1.0 : 통과율 62%
  {'TTR': 0.1473, '평균유사도': 0.9601, '4gram중복률': 0.2377, '길이표준편차': 45.5}

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) 어휘 다양성 (TTR)
ttr = len(set(tokens)) / max(len(tokens), 1)

# 3) 4-gram 중복률
dup_rate = 1 - len(uniq_ng) / max(all_ng, 1)
```

**온도를 바꿨을 때 두 지표가 각각 어떻게 움직이는지** 봅니다.

통과율은 사실 오류가 늘어 온도를 올릴수록 떨어지는 쪽으로 움직입니다. 다만 한 온도에 30건이라 순서가 뒤집혀 보일 수 있습니다. 그럴 때는 건수를 늘려 다시 측정합니다.

다양성 지표는 종류에 따라 다르게 움직입니다. TTR과 4gram 중복률 같은 어휘 지표는 온도를 올리면 나아질 수 있습니다. 교재 3.5가 높은 온도에 맞는다고 한 창의적이고 다양한 응답은 이 지표에서 단어 수준의 변화로 나타납니다. 그런데 **의미를 보는 평균 유사도는 거의 움직이지 않습니다.** 이론 2.6절에서 본 대로 온도가 흔드는 것은 단어 선택이지 다루는 내용이 아닙니다. 통과율을 잃는 만큼 주제의 다양함이 늘지 않으니 온도만 올리면 손해입니다.

다양성은 2.6절의 세 설정으로 올립니다. 이번 실습에서는 프롬프트와 입력 구조를 이미 변주했으므로 더 올리려면 **입력의 폭을 넓힙니다.** 주제 목록을 늘리고 안건 수의 범위를 넓히고 참석자 수를 더 흔듭니다.

평균 유사도 지표가 특히 쓸모 있습니다. 이 값이 0.95를 넘으면 출력이 거의 같아 학습 효과가 떨어집니다. 지금 값이 그보다 낮은지 확인합니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 5. 데이터셋 버전 자동 판별 및 스냅샷 발행 파이프라인
교재 : 4.5.4 지시 파인 튜닝 데이터셋 자동 생성

이론 6.4절의 버전 번호 붙이기를 파이프라인으로 만듭니다. 교재 4.5.4가 권하는 데이터 버전 관리와 롤백입니다.

In [ ]:
# 실습 : 설정 해시(config_hash)로 데이터셋 버전을 식별하고 멱등하게 발행하는 build_dataset 함수를 완성하세요

DATASET_ROOT = Path("./datasets")
DATASET_ROOT.mkdir(exist_ok=True)

if "GEN_PROMPT_V2" not in globals():
    GEN_PROMPT_V2 = GEN_PROMPT
if "generate_batch" not in globals():
    def generate_batch(prompt_tpl, n, start_seed, dist=TARGET_DIST):
        plan_local = []
        for d, r in dist.items():
            plan_local += [d] * max(1, int(n * r))
        random.shuffle(plan_local)
        out = []
        for i, d in enumerate(plan_local):
            m = make_minutes(start_seed + i, d)
            gen = generate(prompt_tpl.format(spec=FORMAT_SPEC, raw=m["raw"]))
            out.append({"id": start_seed + i, "instruction": random.choice(INSTRUCTION_VARIANTS),
                        "input": m["raw"], "output": gen.strip(), "gold": m["gold"],
                        "difficulty": d, "source_seed": start_seed + i})
        return out


def config_hash(cfg):
    s = json.dumps(cfg, sort_keys=True, ensure_ascii=False)
    ### START CODE HERE ### (문자열 s 의 SHA-256 해시 앞 10자리를 반환합니다)
    return hashlib.sha256(s.encode()).hexdigest()[:10]
    ### END CODE HERE ###


def build_dataset(cfg, n=24, start_seed=700000):
    h = config_hash(cfg)
    ver = f"{cfg['version_label']}-{h}"
    out_dir = DATASET_ROOT / ver

    ### START CODE HERE ### (이미 out_dir 디렉터리가 존재하면 재생성하지 않고 기존 card.json을 반환)
    if out_dir.exists():
        print(f"[{ver}] 이미 존재합니다. 재생성하지 않습니다 (멱등성)")
        return ver, json.loads((out_dir / "card.json").read_text(encoding="utf-8"))
    ### END CODE HERE ###

    out_dir.mkdir(parents=True)
    prompt = GEN_PROMPT_V2 if cfg["prompt_version"] == "v2" else GEN_PROMPT

    generated = generate_batch(prompt, n, start_seed, cfg["difficulty_dist"])
    st, rs = run_filters(generated)
    passed = st["passed"]
    if cfg["use_structure_filter"]:
        filtered = [it for it in passed if "결정논의혼동" not in filter_structure(it)]
        if len(filtered) >= 6:
            passed = filtered

    sp = split_by_source(passed) if len(passed) >= 3 else {"train": passed, "valid": [], "test": []}
    stats = {"generated": len(generated),
             "stage_excluded": {k: len(v) for k, v in st.items() if k != "passed"}}
    card = make_dataset_card(cfg["name"], ver, passed, sp, stats)
    card["config"] = cfg

    (out_dir / "card.json").write_text(json.dumps(card, ensure_ascii=False, indent=2),
                                       encoding="utf-8")
    for k, v in sp.items():
        (out_dir / f"{k}.json").write_text(json.dumps(v, ensure_ascii=False, indent=2),
                                           encoding="utf-8")
    print(f"[{ver}] 생성 완료 : {len(passed)}건")
    return ver, card


cfg1 = {"name": "meeting_minutes", "version_label": "v1", "prompt_version": "v1",
        "use_structure_filter": True, "difficulty_dist": TARGET_DIST}
cfg2 = {**cfg1, "version_label": "v2", "prompt_version": "v2"}

ver1, card1 = build_dataset(cfg1, n=24, start_seed=700000)
ver2, card2 = build_dataset(cfg2, n=24, start_seed=800000)

# 재실행해 멱등성 확인
build_dataset(cfg1, n=24, start_seed=700000)

print(f"\n{'항목':<24} {ver1:>18} {ver2:>18}")
print("-" * 64)
for key in ["final_count", "pass_rate"]:
    print(f"{key:<24} {str(card1['stats'][key]):>18} {str(card2['stats'][key]):>18}")
print(f"{'난이도분포':<24} {str(card1['stats']['difficulty_counts']):>18} "
      f"{str(card2['stats']['difficulty_counts']):>18}")

[v1-c7c3b77040] 생성 완료 : 13건
[v2-85b1f76646] 생성 완료 : 11건
[v1-c7c3b77040] 이미 존재합니다. 재생성하지 않습니다 (멱등성)

항목                            v1-c7c3b77040      v2-85b1f76646
----------------------------------------------------------------
final_count                              13                 11
pass_rate                             0.565              0.478
난이도분포                    {'easy': 5, 'normal': 7, 'hard': 1} {'easy': 4, 'normal': 5, 'hard': 2}

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) 설정 해시 계산
return hashlib.sha256(s.encode()).hexdigest()[:10]

# 2) 이미 존재하는 버전이면 재생성 생략 (멱등성)
if out_dir.exists():
    print(f"[{ver}] 이미 존재합니다. 재생성하지 않습니다 (멱등성)")
    return ver, json.loads((out_dir / "card.json").read_text(encoding="utf-8"))
```

**설정 해시로 버전을 만듭니다.** 같은 설정으로 다시 실행하면 같은 버전 식별자가 나오고 이미 있으면 재생성하지 않습니다. 파이프라인 노트의 멱등성 원칙입니다.

카드에 `config`를 넣어 두면 **어떤 설정으로 생성한 데이터셋인지 영구히 남아** 이론 6.4절의 "지난주에 학습한 모델이 더 좋았는데 그때 데이터가 뭐였지" 문제가 풀립니다.

실무에서는 여기에 데이터 자체의 해시도 넣습니다. 설정이 같아도 생성 모델의 비결정성 때문에 데이터가 달라질 수 있으므로 내용 해시가 있어야 완전한 재현 추적이 됩니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 6. 문서 단위 질문 생성(기본 설계)과 청크별 생성(축소 설계) 비교
교재 : 4.5.4 지시 파인 튜닝 데이터셋 자동 생성

이론 2.2절의 두 설계를 모델로 직접 비교합니다. 교재 4.5.4의 기본 설계는 문서 전체에서 질문 목록을 만들고 질문으로 청크를 검색해 답의 근거를 찾습니다. 축소 설계는 검색 없이 청크마다 질문과 답을 함께 만듭니다. 청크별로 만든 쌍에는 4.6의 검색 왕복 검사와 교재의 "검색된 문단에 없는 정보" 검사를 겁니다. 문서는 어려움 회의록 3건을 이어 붙인 900자 안팎이라 교재가 상정한 긴 문서보다 훨씬 짧습니다.

In [ ]:
# 실습 : 문서 단위 질문 생성(A)과 청크별 QA 생성 및 왕복·근거 검사(B)를 비교하세요

QA_SYSTEM = "당신은 사내 회의록으로 평가용 질문을 만드는 담당자입니다. 답변은 반드시 한국어로 작성합니다."

QA_PROMPT = """다음은 사내 회의록의 한 부분입니다. 이 부분만 보고 답할 수 있는 질문 하나와 답을 만드세요.
질문만 읽어도 어느 회의의 어느 안건인지 알 수 있게 회의 날짜와 주제를 넣으세요.
JSON 한 줄로만 출력하세요 : {{"instruction": "질문", "input": "", "output": "답"}}

[회의록 부분]
{chunk}"""

DOC_Q_PROMPT = """다음 회의록 묶음 전체를 읽고 이 문서로 답할 수 있는 질문 {n}개를 만드세요.
JSON 목록으로만 출력하세요 : ["질문1", "질문2", ...]

[회의록 묶음]
{doc}"""


def parse_json(text, kind=dict):
    """모델 출력에서 첫 JSON 객체(또는 목록)를 raw_decode로 안전하게 찾아 읽습니다."""
    cleaned = re.sub(r"```(json)?", "", text).strip()
    pat = r"\{" if kind is dict else r"\["
    decoder = json.JSONDecoder()
    idx = 0
    while idx < len(cleaned):
        m = re.search(pat, cleaned[idx:])
        if not m:
            break
        start = idx + m.start()
        try:
            v, _ = decoder.raw_decode(cleaned, start)
            if isinstance(v, kind):
                return v
        except json.JSONDecodeError:
            pass
        idx = start + 1
    return None


def info(text):
    """답과 청크에서 비교할 정보 : 숫자와 인명"""
    return set(re.findall(r"\d+", text)) | {p for p in PEOPLE if p in text}


docs = [[make_minutes(600000 + 10 * j + t, "hard")["raw"] for t in range(2)] for j in range(2)]
texts = [c for _, c in qa_corpus]
spans = []
for parts in docs:
    start = len(texts)
    texts += [c for raw in parts for c in chunk_minutes(raw)]
    spans.append((start, len(texts)))
t6_search = build_retriever(texts)


def where(i, s, e):
    """청크 i가 그 문서의 앞·중간·뒤 중 어디인지. 다른 문서면 '문서 밖'"""
    return ["앞", "중간", "뒤"][3 * (i - s) // max(e - s, 1)] if s <= i < e else "문서 밖"


rows = {"A 문서 단위": Counter(), "B 청크별": Counter()}
hit = {name: set() for name in rows}
checks, calls = Counter(), 0
for parts, (s, e) in zip(docs, spans):
    qs = parse_json(generate(DOC_Q_PROMPT.format(n=e - s, doc="\n\n".join(parts)),
                             system=QA_SYSTEM, max_new_tokens=400), list) or []
    calls += 1
    for q in qs:                                    # A : 문서 단위 질문 -> 검색 1위 청크
        i = t6_search(str(q), 1)[0]
        rows["A 문서 단위"][where(i, s, e)] += 1
        hit["A 문서 단위"].add(i)
    for ci in range(s, e):                          # B : 청크마다 질의응답(QA) 쌍 하나
        rec = parse_json(generate(QA_PROMPT.format(chunk=texts[ci]), system=QA_SYSTEM,
                                  max_new_tokens=160))
        calls += 1
        if not rec or not rec.get("instruction") or not rec.get("output"):
            checks["JSON 해석 실패"] += 1
            continue
        top = t6_search(str(rec["instruction"]), 3)
        ### START CODE HERE ### (1. 왕복 검색 통과 여부 ci in top 과 2. 근거 밖 정보 차집합 검사)
        is_roundtrip_ok = (ci in top)
        unfaithful_info = info(str(rec["output"])) - info(texts[ci])
        ### END CODE HERE ###
        checks["왕복 통과" if is_roundtrip_ok else "왕복 실패(표시)"] += 1
        if unfaithful_info:
            checks["근거 밖 정보(표시)"] += 1
        rows["B 청크별"][where(ci, s, e)] += 1
        hit["B 청크별"].add(ci)

total = max(sum(e - s for s, e in spans), 1)
print(f"생성 호출 {calls}회 / B 검사 : {dict(checks)}")
print(f"\n{'방식':<12} {'질문':>4} {'앞':>4} {'중간':>4} {'뒤':>4} {'문서 밖':>6} {'덮인 청크':>8}")
for name, cnt in rows.items():
    covered = sum(1 for i in hit[name] if any(s <= i < e for s, e in spans)) / total
    print(f"{name:<12} {sum(cnt.values()):>4} {cnt['앞']:>4} {cnt['중간']:>4} "
          f"{cnt['뒤']:>4} {cnt['문서 밖']:>6} {covered:>8.0%}")

생성 호출 19회 / B 검사 : {'왕복 통과': 15, '왕복 실패(표시)': 2}

방식             질문    앞   중간    뒤   문서 밖    덮인 청크
A 문서 단위        17    3    5    2      7      53%
B 청크별          17    6    6    5      0     100%

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
is_roundtrip_ok = (ci in top)
unfaithful_info = info(str(rec["output"])) - info(texts[ci])
```

**표에서 두 가지를 봅니다.** 첫째, A의 질문이 앞·중간·뒤에 고루 퍼졌는지입니다. 앞에 몰렸다면 이 짧은 길이에서도 쏠림이 나타난 것입니다. 고루 퍼졌어도 이 문서는 교재가 상정한 긴 문서보다 훨씬 짧으므로 긴 문서에서도 그렇다고 결론 내리면 안 됩니다.

둘째, B는 질문을 만든 청크로 셌으므로 JSON을 못 읽은 청크만 덮이지 않습니다. 대신 왕복 실패와 근거 밖 정보가 몇 건인지 봅니다. 왕복에 실패한 질문은 대개 날짜가 빠져 다른 회의록으로 가는 질문이고 근거 밖 정보는 답에 그 답을 만든 청크에 없는 숫자나 이름이 들어간 경우입니다. 둘 다 지우지 않고 표시해 사람이 봅니다.

A는 질문마다 만든 청크가 따로 없어 검색 1위 청크로 셌습니다. 교재 4.5.4의 기본 설계는 여기서 검색된 청크 후보를 LLM에게 주고 가장 알맞은 청크를 골라 답을 완성하게 합니다. 이 트랙은 생성 호출을 60회 안으로 묶으려고 그 단계를 뺐습니다. 답까지 만들려면 질문마다 호출이 한 번씩 늘어납니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 10. 실습 결과 점검과 정리

### Step 10. 결과 점검과 정리

(선택 대상: 트랙 1 학습 데이터 규모별 SFT 성능 변화 측정 / 트랙 2 생성 프롬프트 개선 전후의 통과율 및 학습 성능 비교 / 트랙 3 난이도 구성 비율에 따른 고난도 테스트 성능 비교 / 트랙 4 데이터셋 다양성 지표 측정과 온도별 비교 / 트랙 5 데이터셋 버전 자동 판별 및 스냅샷 발행 파이프라인 / 트랙 6 문서 단위 질문 생성 기본 설계와 청크별 생성 축소 설계 비교)
각자 수행한 두 트랙을 공유합니다. 아래 형식으로 정리해주세요.
1. **선택한 트랙과 이유**
2. **측정 결과** : 1차 규칙·2차 Judge 단계별 통과율, 난이도 재조정(`rebalance`) 전후 분포, 인적 검수 미탐률·오탐률, 그리고 동일 건수(`N_TRAIN=20`) SFT A/B 비교 학습의 평가 점수 변화
3. **합성 데이터 파이프라인의 한계를 만난 지점** : 고난도(`hard`) 시드의 대량 탈락, 온도 상승 시 의미 다양성 정체와 환각 증가, 날짜·금액 계산값에 대한 규칙 필터 오탐, Judge 모델의 판정 한계 등 구체적으로
4. **이 파이프라인을 사내에 배포한다면** : 목표 데이터셋 규모 달성을 위한 생성 예산(초기 생성 배수, 교사·Judge 모델 선정), 인적 검수 표본 비율, 라이선스 및 데이터셋 카드 관리 기준

공유가 끝나면 오늘 만든 것을 한 문장으로 정리해봅시다.
우리는 난이도별 시드와 스키마 강제 디코딩·답변 불가 부정 샘플로 합성 데이터를 만들고, 규칙 필터와 이종 모델 심판·표본 인적 검수의 환류 루프로 불량을 걸러낸 뒤, 출처 단위 분할과 동일 건수 SFT 비교 학습으로 정제 효과를 증명하고 데이터셋 카드를 발행했습니다.
여기서 질문이 하나 생깁니다. 이 모든 것을 통틀어 부르는 이름이 있을까요? 단순히 프롬프트로 텍스트를 대량 찍어내는 생성 스크립트도 아니고 일회성 수작업 라벨링도 아니고, 데이터의 생성·검증·재조정·학습 입증·이력 발행을 하나의 순환 고리로 묶어 모델 성능을 끌어올리는 이 엔지니어링 체계 전체를 말입니다.

<div style="font-size: 18px; line-height: 1.6;">

| 단계 | 핵심 엔지니어링 장치 | 품질 보증 지표 및 산출물 |
| :--- | :--- | :--- |
| **시드 설계 및 생성 (Step 2~3)** | 난이도(`easy`·`normal`·`hard`) 시드 변주 + `GUIDED_JSON_SCHEMA` + 답변 불가(`Refusal`) 부정 샘플 | 구조적 다양성 확보 및 JSON 파싱 실패 원천 차단 |
| **1·2차 자동 필터링 (Step 4)** | 양식·원문 대조·`MinHash` 중복 제거 + 이종 모델(`Llama-3.2`) Judge `예/아니오` 판정 + 왕복 일관성 | 환각·논의/확정 혼동·모호한 질문 제거 |
| **재조정 및 3차 인적 환류 (Step 5~6)** | 고난도 부족분 `rebalance`(`30:50:20`) + 층화·과표집 검수 → `filter_structure` 코드 환류 | 난이도 편향 복원 및 미탐 불량 전수 재차단 |
| **누수 방지 분할 및 SFT 입증 (Step 7~8)** | `source_seed` 그룹 분할(교집합 `0`) + `dataset_card.json` + 동일 건수(`N_TRAIN=20`) QLoRA A/B 학습 | 훈련-테스트 누수 차단 및 정제 데이터 학습 우위 증명 |

</div>

## 마무리 과제. 합성 데이터셋 파이프라인 구축 및 학습 증명 리포트

여러분의 태스크로 데이터셋 생성 파이프라인을 만들고 데이터셋 품질을 학습으로 증명합니다.

**1단계. 태스크 정의**

1. 반복적이고 형식이 정해진 태스크를 정합니다.  
2. 성공 기준을 **기계 검사 가능한 형태로** 4개 이상 작성합니다.  
3. 실패 유형을 4개 이상 미리 적습니다.

**2단계. 생성과 필터**

4. 난이도를 통제할 수 있게 원본 입력을 준비합니다. 난이도는 최소 3단계로 나눕니다.  
5. 200건 이상 생성합니다. 프롬프트 다변화를 적용합니다.  
6. 1차 규칙 필터의 검사를 세 개 이상 만들고 **2차 LLM 판정을 붙여** 단계별 통과율을 기록합니다. judge 모델이 자기 출력에 후한지 스무 건으로 확인합니다.  
7. 난이도 분포를 목표에 맞춰 조정합니다.

**3단계. 검수**

8. 표본 크기를 계산해 근거를 적고 층화 표집으로 표본을 뽑습니다. 적은 계층을 더 뽑았다면 전체 불량률은 원래 비율로 되돌립니다.  
9. 직접 검수해 불량률과 신뢰구간을 산출합니다.  
10. **필터가 놓친 유형을 새 검사로 만들어** 전체에 적용합니다.

**4단계. 데이터셋 발행**

11. 출처 단위로 분할하고 누수를 검증합니다.  
12. 데이터셋 카드를 작성합니다. **조건 확인 다섯 줄**을 넣고 **알려진 한계**를 3개 이상 구체적으로 적습니다.

**5단계. 증명**

13. 큐레이션 유무로 A/B 학습을 합니다. **개수, 하이퍼파라미터, 평가 세트를 고정**해야 합니다.  
14. 난이도별로 나누어 성능을 측정합니다.

**6단계. 리포트**

- **파이프라인 통계표** : 생성 → 각 단계 → 최종의 건수와 통과율  
- **단계별 제외 사유 분포**와 그것으로 판단한 생성 프롬프트의 약점  
- **judge 모델 검증표** : 자기 출력 편향, 불량 적발률, 정상 유지율  
- **검수 결과** : 표본 크기 근거, 불량률과 신뢰구간, 불량 유형, 표시군(검토 대상) 표본의 오탐률  
- **환류** : 검수에서 발견한 유형을 새 검사로 만든 결과와 추가 제외 건수  
- **A/B 비교표** : 난이도별 성능. 차이가 없었다면 원인 추정  
- **데이터셋 카드** : 특히 알려진 한계  
- **다음 버전 계획** : 이번 결과로 드러난 문제와 개선 방향

**평가 기준**

데이터셋을 만들었는지보다 **증명과 정직성**으로 평가합니다.

통과율만 보고하고 학습 결과가 없으면 대리 지표에 머뭅니다. A/B 비교에서 차이가 없었다면 그대로 보고하고 원인을 추정한 리포트가 좋은 리포트입니다.

**알려진 한계를 구체적으로 적었는지**가 실무 역량의 지표입니다. 한계가 없다고 적은 데이터셋 카드는 검토가 부족합니다.